#IOI Ablation Study

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import numpy as np
from pathlib import Path
from transformer_lens import HookedTransformer
from transformer_lens.loading_from_pretrained import convert_gpt2_weights
from transformer_lens import HookedTransformerConfig
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
import json
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import re
def create_ioi_datasets(
    model: HookedTransformer,
    num_samples: int,
    seed: int = 42,
):
    """
    Manually generates diverse clean and corrupted IOI datasets.
    """
    import random
    import torch
    import numpy as np

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    def get_single_token_names(model, names):
        valid = []
        for n in names:
            toks = model.to_tokens(" " + n, prepend_bos=False)
            if toks.shape[1] == 1:
                valid.append(n)
        return valid

    # Single-token names
    RAW_NAMES = [
        "Alice", "Bob", "Charlie", "David", "Emma", "Frank", "Grace", "Henry",
        "Iris", "Jack", "Kate", "Leo", "Mary", "Noah", "Olivia", "Peter",
        "Quinn", "Ruby", "Sam", "Tom", "Uma", "Victor", "Wendy", "Xander",
        "Michael", "Jessica", "Christopher", "Ashley", "Matthew", "Brittany",
        "Josh", "Amanda", "Daniel", "Sarah", "David", "Kimberly",
        "James", "Jennifer", "Robert", "Emily", "John", "Dan", "Sid", "Mark", "Kate", "Jason"
    ]

    OBJECTS = [
        "book", "ball", "pen", "cup", "toy", "gift", "phone", "key",
        "coin", "ring", "watch", "bag", "hat", "shoe"
    ]

    PLACES = [
        "store", "park", "school", "office", "library", "cafe",
        "museum", "garden", "station", "beach"
    ]

    NAMES = get_single_token_names(model, RAW_NAMES)
    print(len(NAMES))
    assert len(NAMES) >= 10, "Too few single-token names!"

    # ✅ Templates are CORRECT - they end with "to"
    TEMPLATES = [
        "Then, {IO} and {S1} went to the {PLACE}. {S2} gave a {OBJ} to",
        "When {IO} and {S1} walked into the {PLACE}, {S2} handed a {OBJ} to",
        "After {IO} and {S1} arrived at the {PLACE}, {S2} passed a {OBJ} to",
        "While {IO} and {S1} were at the {PLACE}, {S2} threw a {OBJ} to",
        "Before {IO} and {S1} left the {PLACE}, {S2} showed a {OBJ} to",
        # Reverse order (S and IO)
        "Then, {S1} and {IO} went to the {PLACE}. {S2} gave a {OBJ} to",
        "When {S1} and {IO} walked into the {PLACE}, {S2} handed a {OBJ} to",
        "After {S1} and {IO} arrived at the {PLACE}, {S2} passed a {OBJ} to",
        "While {S1} and {IO} were at the {PLACE}, {S2} threw a {OBJ} to",
        "Before {S1} and {IO} left the {PLACE}, {S2} showed a {OBJ} to",
        # More variations
        "Since {IO} and {S1} were at the {PLACE}, {S2} offered a {OBJ} to",
        "Though {IO} and {S1} liked the {PLACE}, {S2} sent a {OBJ} to",
        "Once {IO} and {S1} reached the {PLACE}, {S2} brought a {OBJ} to",
        "As {IO} and {S1} entered the {PLACE}, {S2} delivered a {OBJ} to",
        "Because {S1} and {IO} went to the {PLACE}, {S2} took a {OBJ} to",
    ]

    clean_prompts = []
    corrupt_prompts = []
    io_last_token_ids = []
    s_last_token_ids = []

    for _ in range(num_samples):
        template = random.choice(TEMPLATES)
        place = random.choice(PLACES)
        obj = random.choice(OBJECTS)

        # Pick 3 unique names
        io_name, s_name, random_name = random.sample(NAMES, 3)

        # Tokenize names
        io_token = model.to_tokens(" " + io_name, prepend_bos=False)[0, -1].item()
        s_token = model.to_tokens(" " + s_name, prepend_bos=False)[0, -1].item()

        assert io_token < model.cfg.d_vocab
        assert s_token < model.cfg.d_vocab
        io_last_token_ids.append(io_token)
        s_last_token_ids.append(s_token)

        # Clean (ABB) - template ends with "to", then we add IO
        clean_template_filled = template.format(
            IO=io_name, S1=s_name, S2=s_name, PLACE=place, OBJ=obj
        )
        clean_text = clean_template_filled + " " + io_name  # ✅ This is correct

        # Corrupt (ABC) - template ends with "to", then we add IO
        corrupt_template_filled = template.format(
            IO=io_name, S1=s_name, S2=random_name, PLACE=place, OBJ=obj
        )
        corrupt_text = corrupt_template_filled + " " + io_name  # ✅ This is correct

        clean_prompts.append(clean_text)
        corrupt_prompts.append(corrupt_text)

    # Tokenize all prompts
    clean_toks = model.to_tokens(clean_prompts, prepend_bos=True)
    corrupted_toks = model.to_tokens(corrupt_prompts, prepend_bos=True)

    # ✅ NEW CODE (USE THIS INSTEAD):
    end_positions = []

    for i in range(len(clean_prompts)):
        tokens = clean_toks[i]
        io_token = io_last_token_ids[i]

        # Find all occurrences of IO token in this sequence
        io_mask = (tokens == io_token)
        io_positions = io_mask.nonzero(as_tuple=True)[0]

        if len(io_positions) == 0:
            # Fallback if IO token not found (shouldn't happen with single-token names)
            print(f"⚠️ Warning: IO token not found in prompt {i}")
            end_positions.append(len(tokens) - 2)
        else:
            # The last occurrence of IO is the prediction target
            # We want the position RIGHT BEFORE it (where model predicts)
            last_io_pos = io_positions[-1].item()
            end_positions.append(last_io_pos - 1)

    end_pos = torch.tensor(
        end_positions,
        dtype=torch.long,
        device=model.cfg.device
    )


    print(f"\n[VALIDATION] Checking end positions for first 3 examples:")
    for i in range(min(3, len(clean_prompts))):
        end_idx = end_positions[i]

        # Token at end position (what we're conditioning on)
        token_at_end = clean_toks[i][end_idx].item()
        token_at_end_str = model.tokenizer.decode([token_at_end])

        # Token after end position (what model should predict)
        token_after = clean_toks[i][end_idx + 1].item()
        token_after_str = model.tokenizer.decode([token_after])

        # Expected IO name
        io_str = model.tokenizer.decode([io_last_token_ids[i]])

        print(f"  Example {i+1}:")
        print(f"    Full sentence: {clean_prompts[i]}")
        print(f"    End position: {end_idx}")
        print(f"    Token AT end:    '{token_at_end_str}' (position {end_idx})")
        print(f"    Token AFTER end: '{token_after_str}' (position {end_idx + 1})")
        print(f"    Expected IO:     '{io_str}'")

        # Check if they match
        if token_after_str.strip() == io_str.strip():
            print(f"    ✅ CORRECT: Model will predict IO name")
        else:
            print(f"    ❌ ERROR: Mismatch! Check tokenization")
            print(f"       All tokens: {[model.tokenizer.decode([t.item()]) for t in clean_toks[i][:20]]}")

    class DatasetWrapper:
        def __init__(self, toks, io_ids, s_ids, end_pos, texts):
            self.toks = toks
            self.io_tokenIDs = torch.tensor(io_ids, device=model.cfg.device)
            self.s_tokenIDs = torch.tensor(s_ids, device=model.cfg.device)
            self.word_idx = {"end": end_pos}
            self.sentences = texts

    ioi_dataset_final = DatasetWrapper(
        clean_toks, io_last_token_ids, s_last_token_ids, end_pos, clean_prompts
    )

    abc_dataset_final = DatasetWrapper(
        corrupted_toks, io_last_token_ids, s_last_token_ids, end_pos, corrupt_prompts
    )

    # Summary
    unique_count = len(set(clean_prompts))
    print(f"\n✓ Created {num_samples} diverse IOI examples")
    print(f"✓ Unique sentences: {unique_count}/{num_samples}")
    print(f"✓ Templates used: {len(TEMPLATES)}")
    print(f"✓ Sample Clean:   {clean_prompts[0]}")
    print(f"✓ Sample Corrupt: {corrupt_prompts[0]}")

    return ioi_dataset_final, abc_dataset_final

In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_gpt2_weights
import numpy as np
import json, os, csv
from tqdm import tqdm
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader


# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

EXISTING_MODELS_DIR    = "/content/drive/MyDrive/IRP/GPT-2/pruned_models"
ABLATION_SAVE_DIR      = "/content/drive/MyDrive/IRP/ablation_variants_v5"
CACHE_DIR              = "/content/drive/MyDrive/IRP/ablation_cache_v5"
RESULTS_DIR            = "/content/drive/MyDrive/IRP/ablation_results_v5"

SPARSITY_LEVELS = [30, 50, 70]

# IOI circuit heads from your residual stream analysis
PROTECTED_HEADS = {
    8:  [6, 10],
    9:  [6, 9],
    10: [0, 7],
    11: [10],
}


LAYER_BUDGETS_ATTN = {
    0: 0.4,
    1: 0.4,
    2: 0.4,
    3: 0.4,
    4: 0.4,
    5: 0.4,
    6: 0.4,
    7: 0.4,
    8: 0.6,   # raised — layer 8 has heads 6,10 protected, needs budget high enough to reach them
    9: 0.5,   # raised — layer 9 has heads 6,9 protected
    10: 0.4,  # raised — layer 10 has heads 0,7 protected
    11: 0.3,  # raised slightly
}

# Keep original budgets for neurons only
LAYER_BUDGETS_NEURONS = {
    0: 1.50,
    1: 1.50,
    2: 1.40,
    3: 1.30,
    4: 1.10,
    5: 1.00,
    6: 0.90,
    7: 0.85,
    8: 0.75,
    9: 0.55,
    10: 0.50,
    11: 0.35,
}

S_INHIBITION_HEADS = [(8, 6), (8, 10), (9, 6), (9, 9)]

N_LAYERS    = 12
N_HEADS     = 12
HIDDEN_SIZE = 768
HEAD_DIM    = HIDDEN_SIZE // N_HEADS   # 64


# ─────────────────────────────────────────────────────────────
# WEIGHT LOADING  (your verified function)
# ─────────────────────────────────────────────────────────────

def load_model_with_tl(model_path, device, tokenizer=None):
    """Load a pruned HF model into TransformerLens with verified weight transfer."""
    hf_model = GPT2LMHeadModel.from_pretrained(model_path)

    cfg = HookedTransformerConfig(
        n_layers=12, d_model=768, n_heads=12, d_head=64,
        d_mlp=3072, d_vocab=50257, n_ctx=1024,
        act_fn="gelu_new", normalization_type="LN", device=device,
    )

    tl_model = HookedTransformer(cfg)
    tl_state  = convert_gpt2_weights(hf_model, cfg)
    tl_model.load_state_dict(tl_state, strict=False)

    # Verify using MLP weights (reliable for neuron models)
    hf_sp = (hf_model.transformer.h[0].mlp.c_fc.weight == 0).float().mean().item()
    tl_sp = (tl_model.blocks[0].mlp.W_in == 0).float().mean().item()
    print(f"  MLP sparsity — HF: {hf_sp:.2%} | TL: {tl_sp:.2%}", end="")
    print(" ✓" if abs(hf_sp - tl_sp) < 0.01 else " ⚠️  mismatch")

    del hf_model
    torch.cuda.empty_cache()
    tl_model.eval()
    if tokenizer is not None:
        tl_model.set_tokenizer(tokenizer)
    return tl_model


def get_actual_sparsity(model_path):
    m = GPT2LMHeadModel.from_pretrained(model_path)
    total = zeros = 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p.data == 0).sum().item()
    del m
    return zeros / total


# ─────────────────────────────────────────────────────────────
# IMPORTANCE SCORING  (matches original GPT2Pruner logic)
# ─────────────────────────────────────────────────────────────

def calculate_attention_head_importance(attn_layer):
    """
    Matches GPT2Pruner.calculate_attention_head_importance exactly.
    Uses L2-norm of Q-weight slices per head.
    """
    qkv_weight = attn_layer.c_attn.weight   # [hidden_size, 3*hidden_size]
    q_weight   = qkv_weight[:, :HIDDEN_SIZE]
    scores = []
    for i in range(N_HEADS):
        head_w = q_weight[:, i*HEAD_DIM:(i+1)*HEAD_DIM]
        scores.append(torch.norm(head_w, p=2).item())
    return torch.tensor(scores)


def compute_gradient_importance(model, dataloader, device, cache_path=None):
    """Gradient × activation importance, cached to disk."""
    if cache_path and os.path.exists(cache_path):
        print(f"  Loading cached importance from {cache_path}")
        return torch.load(cache_path, map_location="cpu")

    print("  Computing gradient × activation importance...")
    model.train()
    importance = {n: torch.zeros_like(p, device="cpu")
                  for n, p in model.named_parameters() if p.requires_grad}

    for batch in tqdm(dataloader, desc="  Gradient batches"):
        model.zero_grad()
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        out.loss.backward()
        for name, param in model.named_parameters():
            if param.grad is not None:
                importance[name] += (param.grad.abs() * param.data.abs()).cpu()

    model.eval()
    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        torch.save(importance, cache_path)
        print(f"  Saved to {cache_path}")
    return importance


# ─────────────────────────────────────────────────────────────
# PRUNING  (matches original GPT2Pruner._apply_attention_mask)
# ─────────────────────────────────────────────────────────────

def _apply_attention_mask(attn, mask, block_idx, model):
    """
    Identical to GPT2Pruner._apply_attention_mask.
    mask: bool tensor, True = keep, False = prune.
    No bias zeroing — matches original exactly.
    """
    with torch.no_grad():
        for i, keep in enumerate(mask):
            if not keep:
                s = i * HEAD_DIM
                e = s + HEAD_DIM
                attn.c_attn.weight[:, s:e] = 0               # Q
                attn.c_attn.weight[:, HIDDEN_SIZE+s:HIDDEN_SIZE+e] = 0    # K
                attn.c_attn.weight[:, 2*HIDDEN_SIZE+s:2*HIDDEN_SIZE+e] = 0  # V
                attn.c_proj.weight[s:e, :] = 0               # output proj



def prune_attention_heads_ablation(base_model, importance_scores_per_layer,
                                   sparsity_pct, use_layer_budgets,
                                   use_protected_heads):
    nominal = sparsity_pct / 100
    pruned  = deepcopy(base_model)

    MAX_HEADS_PER_LAYER = {
        0: 2, 1: 2, 2: 2, 3: 2,
        4: 3, 5: 3, 6: 3, 7: 3,
        8: 2, 9: 1, 10: 1, 11: 1,
    }

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget = LAYER_BUDGETS_ATTN[layer_idx] if use_layer_budgets else 1.0
            eff    = nominal * budget

            if use_layer_budgets:
                # Full method: budget redistributes pruning + absolute cap
                # prevents catastrophic collapse in any single layer
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    MAX_HEADS_PER_LAYER[layer_idx],
                    int(N_HEADS * 0.40),
                )
            else:
                # No budgets ablation: flat sparsity across all layers
                # looser cap so 30/50/70% actually produce different models
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    int(N_HEADS * 0.70),
                )

            protected = set(PROTECTED_HEADS.get(layer_idx, [])) \
                        if use_protected_heads else set()

            scores = importance_scores_per_layer[layer_idx]
            ranked = sorted(range(N_HEADS), key=lambda h: scores[h].item())

            mask = torch.ones(N_HEADS, dtype=torch.bool)
            pruned_count = 0
            for h in ranked:
                if pruned_count >= n_prune:
                    break
                if h not in protected:
                    mask[h] = False
                    pruned_count += 1

            attn = pruned.transformer.h[layer_idx].attn
            pruned_heads = [h for h in range(N_HEADS) if not mask[h]]
            conflicts = [h for h in pruned_heads if h in protected]
            if conflicts:
                print(f"  Layer {layer_idx}: protection active — blocked heads {conflicts}")
            _apply_attention_mask(attn, mask, layer_idx, pruned)

    return pruned


def compute_layer_gradient_importance(model, dataloader, device, cache_path=None):
    raw = compute_gradient_importance(model, dataloader, device, cache_path)
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        # Change this to use c_proj instead of c_attn
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        if key in raw:
            imp = raw[key]  # [HEAD_DIM * N_HEADS, hidden_size]
            scores = torch.tensor([
                imp[h*HEAD_DIM:(h+1)*HEAD_DIM, :].mean().item()
                for h in range(N_HEADS)
            ])
        else:
            scores = torch.zeros(N_HEADS)
        layer_scores[layer_idx] = scores
    return layer_scores

def compute_layer_gradient_importance_neurons(model, dataloader, device, cache_path=None):
    """
    Returns per-layer neuron importance scores as dict {layer_idx: tensor[MLP_SIZE]}.
    Uses gradient x activation on c_fc weights, matching AttributionPruner exactly.
    """
    raw = compute_gradient_importance(model, dataloader, device, cache_path)

    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.mlp.c_fc.weight"
        if key in raw:
            imp = raw[key]  # [hidden_size, MLP_SIZE]
            # Average gradient importance across input dimension
            scores = imp.mean(dim=0)  # [MLP_SIZE]
        else:
            scores = torch.zeros(3072)
        layer_scores[layer_idx] = scores

    return layer_scores


def prune_neurons_ablation(base_model, importance_scores_per_layer,
                           sparsity_pct, use_layer_budgets):
    """
    Prune neurons using gradient importance with optional layer budgets.

    Note: No head protection equivalent for neurons because your IOI
    circuit analysis identified critical attention heads not neurons.
    The SAE reconstruction error results inform the layer budgets instead.

    importance_scores_per_layer: dict {layer_idx: tensor of shape [MLP_SIZE]}
    """
    MLP_SIZE = 3072
    nominal  = sparsity_pct / 100
    pruned   = deepcopy(base_model)

    # with torch.no_grad():
    #     for layer_idx in range(N_LAYERS):
    #         budget     = LAYER_BUDGETS[layer_idx] if use_layer_budgets else 1.0
    #         eff        = min(nominal * budget, 0.85)
    #         n_prune    = max(0, int(MLP_SIZE * eff))
    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget  = LAYER_BUDGETS_NEURONS[layer_idx] if use_layer_budgets else 1.0
            eff     = min(nominal * budget, 0.85)
            n_prune = max(0, int(MLP_SIZE * eff))
            scores = importance_scores_per_layer[layer_idx]  # [MLP_SIZE]

            # Sort ascending — lowest score pruned first
            ranked = torch.argsort(scores)

            # Build bool mask
            mask = torch.ones(MLP_SIZE, dtype=torch.bool)
            mask[ranked[:n_prune]] = False

            # Apply mask — matches GPT2Pruner and AttributionPruner exactly
            block    = pruned.transformer.h[layer_idx]
            fc_layer = block.mlp.c_fc
            with torch.no_grad():
                fc_layer.weight[:, ~mask] = 0
                if fc_layer.bias is not None:
                    fc_layer.bias[~mask] = 0
                block.mlp.c_proj.weight[~mask, :] = 0

    return pruned


# ─────────────────────────────────────────────────────────────
# PATH HELPERS
# ─────────────────────────────────────────────────────────────

def get_model_path(variant_name, sparsity_pct):
    existing = {
        "magnitude_attention_heads":   EXISTING_MODELS_DIR,
        "magnitude_neurons":           EXISTING_MODELS_DIR,
        "random_attention_heads":      EXISTING_MODELS_DIR,
        "random_neurons":              EXISTING_MODELS_DIR,
    }
    if variant_name in existing:
        return os.path.join(existing[variant_name],
                            f"{variant_name}_sparsity{sparsity_pct}")
    return os.path.join(ABLATION_SAVE_DIR,
                        f"{variant_name}_sparsity{sparsity_pct}")


# ─────────────────────────────────────────────────────────────
# IOI DATASET WRAPPER
# ─────────────────────────────────────────────────────────────

class IOIDatasetWrapper(Dataset):
    def __init__(self, ioi_dataset, tokenizer, max_length=64):
        self.sentences  = ioi_dataset.sentences
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx], return_tensors="pt",
            max_length=self.max_length, truncation=True, padding="max_length",
        )
        ids  = enc["input_ids"].squeeze(0)
        mask = enc["attention_mask"].squeeze(0)
        lbl  = ids.clone()
        lbl[mask == 0] = -100
        return {"input_ids": ids, "attention_mask": mask, "labels": lbl}


# ─────────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────────

def compute_perplexity(model, tokenizer, texts, device,
                       max_length=512, batch_size=8):
    model.eval()
    total_loss = total_tokens = 0
    enc  = tokenizer(texts, return_tensors="pt", truncation=True,
                     max_length=max_length, padding=True)
    ids, mask = enc["input_ids"], enc["attention_mask"]
    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            b_ids  = ids[i:i+batch_size].to(device)
            b_mask = mask[i:i+batch_size].to(device)
            lbl    = b_ids.clone()
            lbl[b_mask == 0] = -100
            out    = model(input_ids=b_ids, attention_mask=b_mask, labels=lbl)
            n      = (lbl != -100).sum().item()
            total_loss   += out.loss.item() * n
            total_tokens += n
    return float(torch.exp(torch.tensor(total_loss / total_tokens)))


def compute_ioi_metrics(model_path, ioi_dataset, abc_dataset,
                        device, tokenizer, n_samples=300):
    tl = load_model_with_tl(model_path, device, tokenizer)
    tl.cfg.use_attn_result = True
    n  = min(n_samples, len(ioi_dataset.sentences))

    # Logit diff + accuracy
    logit_diffs = []
    for i in range(n):
        tokens = tl.to_tokens(ioi_dataset.sentences[i])
        with torch.no_grad():
            logits = tl(tokens)
        end    = ioi_dataset.word_idx["end"][i].item()
        io_tok = ioi_dataset.io_tokenIDs[i]
        s_tok  = ioi_dataset.s_tokenIDs[i]
        logit_diffs.append(
            logits[0, end, io_tok].item() - logits[0, end, s_tok].item())

    mean_ld  = float(np.mean(logit_diffs))
    accuracy = float(np.mean([d > 0 for d in logit_diffs]))

    # S-Inhibition Cohen's d
    patch_effects = []
    for layer, head in S_INHIBITION_HEADS:
        for i in range(min(80, n)):
            clean  = tl.to_tokens(ioi_dataset.sentences[i])
            corr   = tl.to_tokens(abc_dataset.sentences[i])
            end    = ioi_dataset.word_idx["end"][i].item()
            io_tok = ioi_dataset.io_tokenIDs[i]
            s_tok  = ioi_dataset.s_tokenIDs[i]

            with torch.no_grad():
                base_logits = tl(clean)
            base_ld = (base_logits[0, end, io_tok] -
                       base_logits[0, end, s_tok]).item()

            corr_cache = {}
            def save_hook(val, hook, _l=layer, _h=head):
                corr_cache[f"{_l},{_h}"] = val[:, :, _h, :].detach().clone()
                return val

            with torch.no_grad():
                tl.run_with_hooks(
                    corr,
                    fwd_hooks=[(f"blocks.{layer}.attn.hook_result", save_hook)])

            def patch_hook(val, hook, _l=layer, _h=head):
                val[:, :, _h, :] = corr_cache[f"{_l},{_h}"]
                return val

            with torch.no_grad():
                patched = tl.run_with_hooks(
                    clean,
                    fwd_hooks=[(f"blocks.{layer}.attn.hook_result", patch_hook)])

            patch_effects.append(
                base_ld - (patched[0, end, io_tok] -
                           patched[0, end, s_tok]).item())

    arr      = np.array(patch_effects)
    cohens_d = float(arr.mean() / (arr.std() + 1e-8))

    tl.cpu()
    del tl
    torch.cuda.empty_cache()

    return {"mean_logit_diff": mean_ld,
            "accuracy": accuracy,
            "s_inhibition_cohens_d": cohens_d}


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def run_ablation_study(ioi_dataset, abc_dataset, tinystories_texts):

    for d in [ABLATION_SAVE_DIR, CACHE_DIR, RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # Base model
    print("\nEvaluating base model...")
    base_hf  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    base_ppl = compute_perplexity(base_hf, tokenizer, tinystories_texts, DEVICE)
    base_ioi = compute_ioi_metrics(
        os.path.join(EXISTING_MODELS_DIR, "base_model"),
        ioi_dataset, abc_dataset, DEVICE, tokenizer)
    print(f"  Base PPL:     {base_ppl:.2f}")
    print(f"  Base IOI LD:  {base_ioi['mean_logit_diff']:.3f}")
    print(f"  Base IOI Acc: {base_ioi['accuracy']*100:.1f}%")
    print(f"  Base S-Inh d: {base_ioi['s_inhibition_cohens_d']:.3f}")

    # Gradient importance for ablation variants
    ioi_wrapper = IOIDatasetWrapper(ioi_dataset, tokenizer)
    ioi_loader  = DataLoader(ioi_wrapper, batch_size=8, shuffle=False)
    # layer_scores = compute_layer_gradient_importance(
    #     base_hf, ioi_loader, DEVICE,
    #     cache_path=os.path.join(CACHE_DIR, "grad_importance.pt"))
    # Attention head importance scores
    layer_scores_heads = compute_layer_gradient_importance(
        base_hf, ioi_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_heads.pt"))

    # Neuron importance scores
    layer_scores_neurons = compute_layer_gradient_importance_neurons(
        base_hf, ioi_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_neurons.pt"))
    base_hf.cpu()
    torch.cuda.empty_cache()

    variants = [
    # --- Attention head variants (existing) ---
    dict(name="attribution_attention_heads",
         label="Attribution – full (attn heads)",
         load_existing=False,
         component="attention_heads",
         use_layer_budgets=True,
     use_protected_heads=True),
    dict(name="ablation_no_budgets",
         label="Attribution – no layer budgets (attn)",
         load_existing=False,
         component="attention_heads",
         use_layer_budgets=False,
         use_protected_heads=True),
    dict(name="ablation_no_protection",
         label="Attribution – no head protection",
         load_existing=False,
         component="attention_heads",
         use_layer_budgets=True,
         use_protected_heads=False),
    dict(name="ablation_no_budgets_no_protection",
         label="Attribution – no budgets no protection",
         load_existing=False,
         component="attention_heads",
         use_layer_budgets=False,
         use_protected_heads=False),

    # --- Neuron variants (new) ---
    dict(name="attribution_neurons",
         label="Attribution – full (neurons)",
         load_existing=False,
         component="neurons",
         use_layer_budgets=True),
    dict(name="ablation_neurons_no_budgets",
         label="Attribution – no layer budgets (neurons)",
         load_existing=False,
         component="neurons",
         use_layer_budgets=False),

    # --- Baselines ---
    dict(name="magnitude_attention_heads",
         label="Magnitude – attn heads",
         load_existing=True,
         component="attention_heads"),
    dict(name="magnitude_neurons",
         label="Magnitude – neurons",
         load_existing=True,
         component="neurons"),
    dict(name="random_attention_heads",
         label="Random – attn heads",
         load_existing=True,
         component="attention_heads"),
    dict(name="random_neurons",
         label="Random – neurons",
         load_existing=True,
         component="neurons"),
]

    all_results = []

    for v in variants:
        print(f"\n{'='*60}")
        print(f"Variant: {v['label']}")
        print(f"{'='*60}")

        for sparsity_pct in SPARSITY_LEVELS:
            model_path = get_model_path(v["name"], sparsity_pct)
            print(f"  DEBUG path: {model_path}")
            print(f"  DEBUG exists: {os.path.exists(model_path)}")

            # Prune new ablation variants
            if not v["load_existing"] and not os.path.exists(model_path):
                print(f"\n  Pruning sparsity {sparsity_pct}%...")
                base_fresh = GPT2LMHeadModel.from_pretrained("gpt2")
                # pruned = prune_attention_heads_ablation(
                #     base_fresh, layer_scores, sparsity_pct,
                #     use_layer_budgets=v["use_layer_budgets"],
                #     use_protected_heads=v["use_protected_heads"],
                # )
                if v["component"] == "attention_heads":
                  pruned = prune_attention_heads_ablation(
                      base_fresh, layer_scores_heads, sparsity_pct,
                      use_layer_budgets=v["use_layer_budgets"],
                      use_protected_heads=v["use_protected_heads"],
                  )
                elif v["component"] == "neurons":

                  pruned = prune_neurons_ablation(
                      base_fresh, layer_scores_neurons, sparsity_pct,
                      use_layer_budgets=v["use_layer_budgets"],
                  )

                os.makedirs(model_path, exist_ok=True)
                pruned.save_pretrained(model_path)
                tokenizer.save_pretrained(model_path)
                del base_fresh, pruned
                torch.cuda.empty_cache()
                print(f"    Saved to {model_path}")

            if not os.path.exists(model_path):
                print(f"  ⚠️  Not found: {model_path} — skipping")
                continue

            print(f"\n  Evaluating sparsity {sparsity_pct}%...")

            # Perplexity
            hf_model = GPT2LMHeadModel.from_pretrained(model_path).to(DEVICE)
            ppl      = compute_perplexity(hf_model, tokenizer,
                                          tinystories_texts, DEVICE)
            hf_model.cpu()
            del hf_model
            torch.cuda.empty_cache()

            actual = get_actual_sparsity(model_path)
            ioi    = compute_ioi_metrics(model_path, ioi_dataset,
                                         abc_dataset, DEVICE, tokenizer)

            row = {
                "variant":             v["label"],
                "nominal_sparsity":    sparsity_pct,
                "actual_sparsity_pct": round(actual * 100, 1),
                "perplexity":          round(ppl, 2),
                "ppl_delta_pct":       round((ppl - base_ppl) / base_ppl * 100, 1),
                "ioi_logit_diff":      round(ioi["mean_logit_diff"], 3),
                "ioi_accuracy_pct":    round(ioi["accuracy"] * 100, 1),
                "s_inhibition_d":      round(ioi["s_inhibition_cohens_d"], 3),
            }
            all_results.append(row)

            print(f"    Actual sparsity:  {actual*100:.1f}%")
            print(f"    Perplexity:       {ppl:.1f} (+{row['ppl_delta_pct']}%)")
            print(f"    IOI logit diff:   {ioi['mean_logit_diff']:.3f}")
            print(f"    IOI accuracy:     {ioi['accuracy']*100:.1f}%")
            print(f"    S-Inh Cohen's d:  {ioi['s_inhibition_cohens_d']:.3f}")

    # Save
    os.makedirs(RESULTS_DIR, exist_ok=True)
    csv_path = os.path.join(RESULTS_DIR, "ablation_results.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)
    print(f"\n✓ Saved to {csv_path}")

    with open(os.path.join(RESULTS_DIR, "base_metrics.json"), "w") as f:
        json.dump({
            "ppl": base_ppl,
            "ioi_ld": base_ioi["mean_logit_diff"],
            "ioi_acc": base_ioi["accuracy"],
            "s_inh_d": base_ioi["s_inhibition_cohens_d"],
        }, f, indent=2)

    print_table(all_results, base_ppl, base_ioi)
    return all_results


# ─────────────────────────────────────────────────────────────
# TABLE
# ─────────────────────────────────────────────────────────────

def print_table(results, base_ppl, base_ioi):
    print(f"\n{'='*95}")
    print("ABLATION STUDY RESULTS")
    print(f"Base: PPL={base_ppl:.1f} | IOI LD={base_ioi['mean_logit_diff']:.3f} | "
          f"IOI Acc={base_ioi['accuracy']*100:.1f}% | "
          f"S-Inh d={base_ioi['s_inhibition_cohens_d']:.3f}")
    print(f"{'='*95}")
    for sparsity_pct in SPARSITY_LEVELS:
        rows = [r for r in results if r["nominal_sparsity"] == sparsity_pct]
        print(f"\n── {sparsity_pct}% Nominal Sparsity ──")
        print(f"{'Variant':<38} {'Act%':>5} {'PPL':>8} {'PPLΔ%':>7} "
              f"{'IOI LD':>8} {'IOI Acc':>8} {'S-Inh d':>9}")
        print("-" * 88)
        for r in rows:
            print(f"{r['variant']:<38} "
                  f"{r['actual_sparsity_pct']:>5.1f} "
                  f"{r['perplexity']:>8.1f} "
                  f"{r['ppl_delta_pct']:>6.0f}% "
                  f"{r['ioi_logit_diff']:>8.3f} "
                  f"{r['ioi_accuracy_pct']:>7.1f}% "
                  f"{r['s_inhibition_d']:>9.3f}")



In [ ]:
if __name__ == "__main__":
    from transformer_lens import HookedTransformer
    from datasets import load_dataset

    tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    ioi_dataset, abc_dataset = create_ioi_datasets(
        model=tl_model, num_samples=500, seed=42)
    tl_model.cpu()
    del tl_model
    torch.cuda.empty_cache()

    ts = load_dataset("roneneldan/TinyStories",
                      split="validation", streaming=True)
    tinystories_texts = [ex["text"] for ex in list(ts.take(200))]

    results = run_ablation_study(ioi_dataset, abc_dataset, tinystories_texts)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained model gpt2 into HookedTransformer
45

[VALIDATION] Checking end positions for first 3 examples:
  Example 1:
    Full sentence: Since Ruby and Peter were at the park, Peter offered a book to Ruby
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Ruby' (position 15)
    Expected IO:     ' Ruby'
    ✅ CORRECT: Model will predict IO name
  Example 2:
    Full sentence: After Kimberly and Frank arrived at the park, Frank passed a watch to Kimberly
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Kimberly' (position 15)
    Expected IO:     ' Kimberly'
    ✅ CORRECT: Model will predict IO name
  Example 3:
    Full sentence: When Noah and Frank walked into the store, Noah handed a book to Frank
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Frank' (position 15)
    Expected IO:     ' Frank'
    ✅ CORRECT: Model will predict IO name

✓ Created 500 diverse IOI example

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  MLP sparsity — HF: 50.00% | TL: 50.00% ✓
Moving model to device:  cpu
    Actual sparsity:  22.8%
    Perplexity:       275.0 (+2645.7%)
    IOI logit diff:   -0.342
    IOI accuracy:     42.0%
    S-Inh Cohen's d:  0.159
  DEBUG path: /content/drive/MyDrive/IRP/GPT-2/pruned_models/random_neurons_sparsity70
  DEBUG exists: True

  Evaluating sparsity 70%...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  MLP sparsity — HF: 69.99% | TL: 69.99% ✓
Moving model to device:  cpu
    Actual sparsity:  31.9%
    Perplexity:       1095.6 (+10839.2%)
    IOI logit diff:   -0.038
    IOI accuracy:     45.3%
    S-Inh Cohen's d:  -0.229

✓ Saved to /content/drive/MyDrive/IRP/ablation_results_v5/ablation_results.csv

ABLATION STUDY RESULTS
Base: PPL=10.0 | IOI LD=2.899 | IOI Acc=95.3% | S-Inh d=0.501

── 30% Nominal Sparsity ──
Variant                                 Act%      PPL   PPLΔ%   IOI LD  IOI Acc   S-Inh d
----------------------------------------------------------------------------------------
Attribution – full (attn heads)          2.1     43.1    330%    4.753    82.7%     0.507
Attribution – no layer budgets (attn)    5.7    329.9   3194%   -1.772    35.0%    -0.033
Attribution – no head protection         2.1     43.1    330%    4.753    82.7%     0.507
Attribution – no budgets no protection   5.7    329.9   3194%   -1.772    35.0%    -0.033
Attribution – full (neurons)            

In [ ]:
layer_scores_heads = compute_layer_gradient_importance(
    None, None, DEVICE,
    cache_path="/content/drive/MyDrive/IRP/ablation_cache_v5/grad_importance_heads.pt")

for layer_idx in [8, 9, 10, 11]:
    scores = layer_scores_heads[layer_idx]
    ranked = sorted(range(12), key=lambda h: scores[h].item())
    protected = list(PROTECTED_HEADS.get(layer_idx, []))
    print(f"Layer {layer_idx}: ranked low→high = {ranked}")
    print(f"           protected = {protected}")
    print(f"           protected rank positions = {[ranked.index(h) for h in protected]}")
    print()

  Loading cached importance from /content/drive/MyDrive/IRP/ablation_cache_v5/grad_importance_heads.pt
Layer 8: ranked low→high = [1, 11, 0, 2, 4, 8, 7, 9, 3, 6, 5, 10]
           protected = [6, 10]
           protected rank positions = [9, 11]

Layer 9: ranked low→high = [1, 11, 2, 8, 0, 5, 6, 9, 10, 4, 7, 3]
           protected = [6, 9]
           protected rank positions = [6, 7]

Layer 10: ranked low→high = [8, 9, 4, 3, 11, 1, 5, 6, 10, 2, 0, 7]
           protected = [0, 7]
           protected rank positions = [10, 11]

Layer 11: ranked low→high = [9, 5, 7, 6, 10, 3, 4, 2, 1, 11, 0, 8]
           protected = [10]
           protected rank positions = [4]



#Ablation Greater Than

In [ ]:

import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from dataclasses import dataclass
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import get_act_name
import random

def generate_real_sentence(noun: str, year: int, eos: bool = False) -> str:
    """Generate clean sentence: model should predict YY digits > current YY."""
    century = year // 100
    sentence = f"The {noun} lasted from the year {year} to the year {century}"
    if eos:
        sentence = "<|endoftext|> " + sentence
    return sentence


def generate_bad_sentence(noun: str, year: int, eos: bool = False) -> str:
    """Generate corrupted sentence: violates greater-than (YY = 01)."""
    century = year // 100
    sentence = f"The {noun} lasted from the year {century}01 to the year {century}"
    if eos:
        sentence = "<|endoftext|> " + sentence
    return sentence


def is_valid_year(year: str, tokenizer) -> bool:
    """Check if year tokenizes to exactly 2 tokens (century + YY)."""
    _year = " " + year
    token = tokenizer(_year)["input_ids"]
    detok = tokenizer.convert_ids_to_tokens(token)
    return len(detok) == 2 and len(detok[1]) == 2


class YearDataset:
    """
    Authentic dataset class from Hanna et al. (2023).

    Task: "The {noun} lasted from the year {XXYY} to the year {XX}"
    Model should predict: YY digits where YY > current_YY

    Clean (good): XXYY is a real year (e.g., 1763)
    Corrupted (bad): XX01 (forces YY=01, which is always wrong)
    """

    def __init__(
        self,
        years_to_sample_from: torch.Tensor,
        N: int,
        nouns: Union[str, List[str], Path],
        tokenizer,
        balanced: bool = True,
        eos: bool = False,
        device: str = "cpu",
    ):
        """
        Args:
            years_to_sample_from: Tensor of valid years (e.g., 1000-1999)
            N: Number of samples
            nouns: List of nouns or path to noun file
            tokenizer: HuggingFace tokenizer
            balanced: If True, balance distribution of YY values
            eos: If True, prepend <|endoftext|> token
            device: Device for tensors
        """
        self.years_to_sample_from = years_to_sample_from
        self.N = N
        self.eos = eos
        self.tokenizer = tokenizer

        # Handle nouns input
        if isinstance(nouns, str):
            noun_list = [nouns]
        elif isinstance(nouns, list):
            noun_list = nouns
        elif isinstance(nouns, Path):
            with open(nouns, "r") as f:
                noun_list = [line.strip() for line in f]
        else:
            raise ValueError(f"Got bad type of nouns: {type(nouns)}")

        self.nouns = random.choices(noun_list, k=N)

        # Sample years (balanced across YY values for better coverage)
        if balanced:
            years = []
            current_year = 2
            years_to_sample_from_YY = self.years_to_sample_from % 100
            for i in range(N):
                sample_pool = self.years_to_sample_from[years_to_sample_from_YY == current_year]
                if len(sample_pool) > 0:
                    years.append(sample_pool[random.randrange(len(sample_pool))].item())
                else:
                    # Fallback if no years with this YY value
                    years.append(self.years_to_sample_from[random.randrange(len(self.years_to_sample_from))].item())
                current_year += 1
                if current_year >= 99:
                    current_year -= 97
            self.years = torch.tensor(years)
        else:
            indices = torch.randint(0, len(self.years_to_sample_from), (N,))
            self.years = self.years_to_sample_from[indices]

        self.years_XX = self.years // 100
        self.years_YY = self.years % 100

        # Generate sentences
        self.good_sentences = [
            generate_real_sentence(noun, int(year.item()), eos=eos)
            for noun, year in zip(self.nouns, self.years)
        ]
        self.bad_sentences = [
            generate_bad_sentence(noun, int(year.item()), eos=eos)
            for noun, year in zip(self.nouns, self.years)
        ]

        # Tokenize
        good_tokenized = tokenizer(self.good_sentences, return_tensors="pt", padding=True)
        self.good_toks = good_tokenized["input_ids"]
        good_attn = good_tokenized["attention_mask"]

        bad_tokenized = tokenizer(self.bad_sentences, return_tensors="pt", padding=True)
        self.bad_toks = bad_tokenized["input_ids"]
        bad_attn = bad_tokenized["attention_mask"]

        # Create mask for valid predictions (YY > current_YY)
        _good_logits_masks = []
        for year in self.years_YY:
            logits_mask = torch.arange(100)
            _good_logits_masks.append(logits_mask > year)
        self.good_mask = torch.stack(_good_logits_masks)

        # Move to device
        self.good_toks = self.good_toks.to(device)
        self.bad_toks = self.bad_toks.to(device)
        self.good_mask = self.good_mask.to(device)

    def __len__(self):
        return self.N


def create_greater_than_datasets(
    model: HookedTransformer,
    num_samples: int = 1000,
    year_range: Tuple[int, int] = (1000, 1999),
    seed: int = 42,
    balanced: bool = True,
    eos: bool = False
):
    """
    Create Greater-Than datasets using authentic Hanna et al. (2023) implementation.

    Task format: "The {noun} lasted from the year {XXYY} to the year {XX}"
    Model should predict: YY digits where YY > current_YY

    Args:
        model: HookedTransformer model
        num_samples: Number of examples to generate
        year_range: (min_year, max_year) for sampling (default: 1000-1999)
        seed: Random seed
        balanced: Balance distribution of YY values (recommended True)
        eos: Prepend <|endoftext|> token

    Returns:
        Tuple of (clean_dataset, corrupted_dataset)
        - Clean (good): Real years like 1763
        - Corrupted (bad): Forces YY=01 (always wrong)
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # Create tensor of valid years
    min_year, max_year = year_range
    years_to_sample = torch.arange(min_year, max_year + 1)

    # Default nouns from the paper
    DEFAULT_NOUNS = [
        "war", "event", "period", "era", "age", "epoch", "reign", "dynasty",
        "empire", "kingdom", "republic", "civilization", "movement", "revolution",
        "conflict", "battle", "campaign", "expedition", "voyage", "journey"
    ]

    # Create dataset using authentic implementation
    dataset = YearDataset(
        years_to_sample_from=years_to_sample,
        N=num_samples,
        nouns=DEFAULT_NOUNS,
        tokenizer=model.tokenizer,
        balanced=balanced,
        eos=eos,
        device=model.cfg.device
    )

    # ✅ FIX: Find end positions correctly
    # The model predicts AFTER "to the year XX", so we need to find where XX token ends
    end_positions = []

    for i in range(num_samples):
        toks = dataset.good_toks[i]
        sentence = dataset.good_sentences[i]

        # The end position is the last non-padding token
        # (The sentence ends with "XX" and model predicts "YY" next)

        # Find last non-padding token
        if hasattr(model.tokenizer, 'pad_token_id') and model.tokenizer.pad_token_id is not None:
            # Find last non-pad token
            non_pad_mask = toks != model.tokenizer.pad_token_id
            if non_pad_mask.any():
                last_real_token_idx = non_pad_mask.nonzero()[-1].item()
            else:
                last_real_token_idx = len(toks) - 1
        else:
            # No padding, just use last token
            last_real_token_idx = len(toks) - 1

        # The prediction position is the last real token position
        # (model predicts NEXT token after this)
        end_positions.append(last_real_token_idx)

    end_pos = torch.tensor(end_positions, dtype=torch.long, device=model.cfg.device)

    # ✅ FIX: Get answer tokens correctly
    # We need to find which tokens correspond to valid YY values
    answer_tokens = []

    for i in range(num_samples):
        current_yy = dataset.years_YY[i].item()

        # Use a representative valid YY value for backward compatibility
        # In practice, we'll check ALL valid YY in the evaluation
        representative_yy = min(current_yy + 20, 99)

        # Tokenize this YY value AS IT APPEARS AFTER THE CENTURY (NO SPACE)
        # The model sees: "...to the year 17" and next token is "65" (not " 65")
        yy_str = f"{representative_yy:02d}"

        # ✅ CRITICAL: NO leading space!
        yy_tokens = model.to_tokens(yy_str, prepend_bos=False)[0]

        if len(yy_tokens) == 1:
            answer_tokens.append(yy_tokens[0].item())
        else:
            # Multi-token - use LAST token (e.g., "02" → ['0', '2'], use '2')
            answer_tokens.append(yy_tokens[-1].item())

    # ✅ ENHANCED VALIDATION
    print(f"\n[VALIDATION] Greater-Than Dataset (Hanna et al. 2023 authentic):")
    print(f"  Generated {num_samples} examples")
    print(f"  Year range: {min_year}-{max_year}")
    print(f"  Balanced: {balanced}")

    for i in range(min(3, num_samples)):
        year = dataset.years[i].item()
        xx = dataset.years_XX[i].item()
        yy = dataset.years_YY[i].item()

        print(f"\n  Example {i+1}:")
        print(f"    Year: {year} (XX={xx}, YY={yy})")
        print(f"    Noun: {dataset.nouns[i]}")
        print(f"    Clean: {dataset.good_sentences[i]}")
        print(f"    Corrupt: {dataset.bad_sentences[i]}")

        # Show actual tokens
        toks = dataset.good_toks[i]
        print(f"    Tokens: {model.tokenizer.decode(toks)}")
        print(f"    End position: {end_positions[i]}")
        print(f"    Token at end: '{model.tokenizer.decode([toks[end_positions[i]].item()])}'")

        # Show what comes next (should be YY or padding)
        if end_positions[i] + 1 < len(toks):
            next_token = toks[end_positions[i] + 1].item()
            print(f"    Next token: '{model.tokenizer.decode([next_token])}'")

        print(f"    Valid YY range: {yy+1} to 99")
        print(f"    Representative answer token: {answer_tokens[i]}")
        print(f"    Decodes to: '{model.tokenizer.decode([answer_tokens[i]])}'")

    # Check tokenization of years
    print(f"\n  Tokenization check:")
    sample_year = dataset.years[0].item()
    sample_yy = f"{dataset.years_YY[0].item():02d}"

    # Check how the full year tokenizes
    full_year_tokens = model.to_tokens(f" {sample_year}", prepend_bos=False)[0]
    print(f"    Full year ' {sample_year}' tokenizes to: {[model.tokenizer.decode([t.item()]) for t in full_year_tokens]}")

    # Check how YY tokenizes WITHOUT space (as it appears after century)
    yy_tokens_no_space = model.to_tokens(sample_yy, prepend_bos=False)[0]
    print(f"    YY '{sample_yy}' (no space) tokenizes to: {[model.tokenizer.decode([t.item()]) for t in yy_tokens_no_space]}")

    # Check how YY tokenizes WITH space (for comparison)
    yy_tokens_with_space = model.to_tokens(f" {sample_yy}", prepend_bos=False)[0]
    print(f"    YY ' {sample_yy}' (with space) tokenizes to: {[model.tokenizer.decode([t.item()]) for t in yy_tokens_with_space]}")

    is_valid = is_valid_year(str(sample_year), model.tokenizer)
    print(f"    Year {sample_year} is_valid check: {is_valid}")

    print(f"\n  ✅ CRITICAL: After 'to the year {dataset.years_XX[0].item()}', next token is '{sample_yy}' (NO space)")

    # Wrapper class for compatibility with patching code
    class DatasetWrapper:
        def __init__(self, dataset, answer_tokens, end_pos, is_clean=True):
            self.toks = dataset.good_toks if is_clean else dataset.bad_toks
            self.answer_tokenIDs = torch.tensor(answer_tokens, device=model.cfg.device)
            self.years = dataset.years
            self.years_XX = dataset.years_XX
            self.years_YY = dataset.years_YY
            self.word_idx = {"end": end_pos}
            self.sentences = dataset.good_sentences if is_clean else dataset.bad_sentences
            self.nouns = dataset.nouns
            self.good_mask = dataset.good_mask  # Mask for valid YY values
            self.tokenizer = model.tokenizer

    clean_dataset = DatasetWrapper(dataset, answer_tokens, end_pos, is_clean=True)
    corrupt_dataset = DatasetWrapper(dataset, answer_tokens, end_pos, is_clean=False)

    print(f"\n✓ Created Greater-Than datasets successfully")

    return clean_dataset, corrupt_dataset

In [ ]:
"""
Ablation Study — Greater-Than Circuit
======================================
Mirrors the IOI ablation study exactly but uses:
- Greater-Than prompts for importance score computation
- GT accuracy and GT prob mass as evaluation metrics
- GT circuit heads for explicit protection (Hanna et al. 2023)
- Same pruning logic, budgets, and variant structure as v5

GT circuit heads (from Hanna et al. 2023):
    year_difference:   L9H1  (most critical — directly computes YY comparison)
    year_encoding:     L7H11, L5H0
    output_computation: L7H10, L8H6
"""

import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_gpt2_weights
import numpy as np
import json, os, csv
from tqdm import tqdm
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader
from scipy.stats import ttest_rel, wilcoxon
from statsmodels.stats.multitest import multipletests
from typing import Tuple, Dict, Union


# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXISTING_MODELS_DIR = "/content/drive/MyDrive/IRP/GPT-2/pruned_models"
ABLATION_SAVE_DIR   = "/content/drive/MyDrive/IRP/ablation_variants_gt"
CACHE_DIR           = "/content/drive/MyDrive/IRP/ablation_cache_gt"
RESULTS_DIR         = "/content/drive/MyDrive/IRP/ablation_results_gt"

SPARSITY_LEVELS = [30, 50, 70]

# GT circuit heads from Hanna et al. (2023) — used for explicit protection
# These are the heads your residual stream decomposition identified as critical
GT_PROTECTED_HEADS = {
    5:  [0],    # year_encoding
    7:  [10, 11], # year_encoding + output_computation
    8:  [6],    # output_computation
    9:  [1],    # year_difference — most critical
}

# GT comparison heads for patching evaluation
# year_difference head is the primary patching target
GT_COMPARISON_HEADS = [(9, 1)]

# Same layer budgets as IOI ablation — CKA findings are circuit-agnostic
LAYER_BUDGETS_ATTN = {
    0: 0.4,
    1: 0.4,
    2: 0.4,
    3: 0.4,
    4: 0.4,
    5: 0.4,
    6: 0.4,
    7: 0.4,
    8: 0.6,
    9: 0.5,
    10: 0.4,
    11: 0.3,
}

LAYER_BUDGETS_NEURONS = {
    0: 1.50,
    1: 1.50,
    2: 1.40,
    3: 1.30,
    4: 1.10,
    5: 1.00,
    6: 0.90,
    7: 0.85,
    8: 0.75,
    9: 0.55,
    10: 0.50,
    11: 0.35,
}

N_LAYERS    = 12
N_HEADS     = 12
HIDDEN_SIZE = 768
HEAD_DIM    = HIDDEN_SIZE // N_HEADS  # 64
MLP_SIZE    = 3072


# ─────────────────────────────────────────────────────────────
# WEIGHT LOADING
# ─────────────────────────────────────────────────────────────

def load_model_with_tl(model_path, device, tokenizer=None):
    hf_model = GPT2LMHeadModel.from_pretrained(model_path)

    cfg = HookedTransformerConfig(
        n_layers=12, d_model=768, n_heads=12, d_head=64,
        d_mlp=3072, d_vocab=50257, n_ctx=1024,
        act_fn="gelu_new", normalization_type="LN", device=device,
    )

    tl_model = HookedTransformer(cfg)
    tl_state  = convert_gpt2_weights(hf_model, cfg)
    tl_model.load_state_dict(tl_state, strict=False)

    hf_sp = (hf_model.transformer.h[0].mlp.c_fc.weight == 0).float().mean().item()
    tl_sp = (tl_model.blocks[0].mlp.W_in == 0).float().mean().item()
    print(f"  MLP sparsity — HF: {hf_sp:.2%} | TL: {tl_sp:.2%}", end="")
    print(" ✓" if abs(hf_sp - tl_sp) < 0.01 else " ⚠️  mismatch")

    del hf_model
    torch.cuda.empty_cache()
    tl_model.eval()
    if tokenizer is not None:
        tl_model.set_tokenizer(tokenizer)
    return tl_model


def get_actual_sparsity(model_path):
    m = GPT2LMHeadModel.from_pretrained(model_path)
    total = zeros = 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p.data == 0).sum().item()
    del m
    return zeros / total


# ─────────────────────────────────────────────────────────────
# GT DATASET WRAPPER
# Wraps Greater-Than clean sentences for importance scoring.
# Equivalent to IOIDatasetWrapper but uses GT sentences.
# ─────────────────────────────────────────────────────────────

class GTDatasetWrapper(Dataset):
    def __init__(self, gt_dataset, tokenizer, max_length=64):
        # gt_dataset is the clean DatasetWrapper returned by create_greater_than_datasets
        self.sentences  = gt_dataset.sentences
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx], return_tensors="pt",
            max_length=self.max_length, truncation=True, padding="max_length",
        )
        ids  = enc["input_ids"].squeeze(0)
        mask = enc["attention_mask"].squeeze(0)
        lbl  = ids.clone()
        lbl[mask == 0] = -100
        return {"input_ids": ids, "attention_mask": mask, "labels": lbl}


# ─────────────────────────────────────────────────────────────
# IMPORTANCE SCORING
# ─────────────────────────────────────────────────────────────

def compute_gradient_importance(model, dataloader, device, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print(f"  Loading cached importance from {cache_path}")
        return torch.load(cache_path, map_location="cpu")

    print("  Computing gradient × activation importance...")
    model.train()
    importance = {n: torch.zeros_like(p, device="cpu")
                  for n, p in model.named_parameters() if p.requires_grad}

    for batch in tqdm(dataloader, desc="  Gradient batches"):
        model.zero_grad()
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        out.loss.backward()
        for name, param in model.named_parameters():
            if param.grad is not None:
                importance[name] += (param.grad.abs() * param.data.abs()).cpu()

    model.eval()
    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        torch.save(importance, cache_path)
        print(f"  Saved to {cache_path}")
    return importance


def compute_layer_gradient_importance(model, dataloader, device, cache_path=None):
    raw = compute_gradient_importance(model, dataloader, device, cache_path)
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        if key in raw:
            imp = raw[key]  # [HEAD_DIM * N_HEADS, hidden_size]
            scores = torch.tensor([
                imp[h*HEAD_DIM:(h+1)*HEAD_DIM, :].mean().item()
                for h in range(N_HEADS)
            ])
        else:
            scores = torch.zeros(N_HEADS)
        layer_scores[layer_idx] = scores
    return layer_scores


def compute_layer_gradient_importance_neurons(model, dataloader, device, cache_path=None):
    raw = compute_gradient_importance(model, dataloader, device, cache_path)
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.mlp.c_fc.weight"
        if key in raw:
            imp = raw[key]  # [hidden_size, MLP_SIZE]
            scores = imp.mean(dim=0)  # [MLP_SIZE]
        else:
            scores = torch.zeros(MLP_SIZE)
        layer_scores[layer_idx] = scores
    return layer_scores


def run_gt_head_diagnostic(layer_scores_heads):
    """
    Check whether GT circuit heads naturally rank highly on GT importance scores.
    Mirrors the IOI diagnostic that showed implicit protection.
    Rank position 11 = most important, 0 = least important (pruned first).
    """
    print("\n" + "="*60)
    print("GT CIRCUIT HEAD DIAGNOSTIC")
    print("Do GT importance scores implicitly protect GT circuit heads?")
    print("="*60)

    for layer_idx, heads in GT_PROTECTED_HEADS.items():
        scores = layer_scores_heads[layer_idx]
        ranked = sorted(range(N_HEADS), key=lambda h: scores[h].item())
        print(f"\nLayer {layer_idx}: ranked low→high = {ranked}")
        print(f"           GT circuit heads = {heads}")
        print(f"           rank positions   = {[ranked.index(h) for h in heads]}")
        print(f"           (11=most important, 0=least important/pruned first)")


# ─────────────────────────────────────────────────────────────
# PRUNING
# ─────────────────────────────────────────────────────────────

def _apply_attention_mask(attn, mask):
    with torch.no_grad():
        for i, keep in enumerate(mask):
            if not keep:
                s = i * HEAD_DIM
                e = s + HEAD_DIM
                attn.c_attn.weight[:, s:e] = 0
                attn.c_attn.weight[:, HIDDEN_SIZE+s:HIDDEN_SIZE+e] = 0
                attn.c_attn.weight[:, 2*HIDDEN_SIZE+s:2*HIDDEN_SIZE+e] = 0
                attn.c_proj.weight[s:e, :] = 0


def prune_attention_heads_ablation(base_model, importance_scores_per_layer,
                                   sparsity_pct, use_layer_budgets,
                                   use_protected_heads):
    nominal = sparsity_pct / 100
    pruned  = deepcopy(base_model)

    MAX_HEADS_PER_LAYER = {
        0: 2, 1: 2, 2: 2, 3: 2,
        4: 3, 5: 3, 6: 3, 7: 3,
        8: 2, 9: 1, 10: 1, 11: 1,
    }

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget = LAYER_BUDGETS_ATTN[layer_idx] if use_layer_budgets else 1.0
            eff    = nominal * budget

            if use_layer_budgets:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    MAX_HEADS_PER_LAYER[layer_idx],
                    int(N_HEADS * 0.40),
                )
            else:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    int(N_HEADS * 0.70),
                )

            # Use GT protected heads instead of IOI protected heads
            protected = set(GT_PROTECTED_HEADS.get(layer_idx, [])) \
                        if use_protected_heads else set()

            scores = importance_scores_per_layer[layer_idx]
            ranked = sorted(range(N_HEADS), key=lambda h: scores[h].item())

            mask = torch.ones(N_HEADS, dtype=torch.bool)
            pruned_count = 0
            for h in ranked:
                if pruned_count >= n_prune:
                    break
                if h not in protected:
                    mask[h] = False
                    pruned_count += 1

            attn = pruned.transformer.h[layer_idx].attn
            _apply_attention_mask(attn, mask)

    return pruned


def prune_neurons_ablation(base_model, importance_scores_per_layer,
                           sparsity_pct, use_layer_budgets):
    nominal = sparsity_pct / 100
    pruned  = deepcopy(base_model)

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget  = LAYER_BUDGETS_NEURONS[layer_idx] if use_layer_budgets else 1.0
            eff     = min(nominal * budget, 0.85)
            n_prune = max(0, int(MLP_SIZE * eff))

            scores = importance_scores_per_layer[layer_idx]
            ranked = torch.argsort(scores)

            mask = torch.ones(MLP_SIZE, dtype=torch.bool)
            mask[ranked[:n_prune]] = False

            block    = pruned.transformer.h[layer_idx]
            fc_layer = block.mlp.c_fc
            with torch.no_grad():
                fc_layer.weight[:, ~mask] = 0
                if fc_layer.bias is not None:
                    fc_layer.bias[~mask] = 0
                block.mlp.c_proj.weight[~mask, :] = 0

    return pruned


# ─────────────────────────────────────────────────────────────
# PATH HELPERS
# ─────────────────────────────────────────────────────────────

def get_model_path(variant_name, sparsity_pct):
    existing = {
        "magnitude_attention_heads": EXISTING_MODELS_DIR,
        "magnitude_neurons":         EXISTING_MODELS_DIR,
        "random_attention_heads":    EXISTING_MODELS_DIR,
        "random_neurons":            EXISTING_MODELS_DIR,
    }
    if variant_name in existing:
        return os.path.join(existing[variant_name],
                            f"{variant_name}_sparsity{sparsity_pct}")
    return os.path.join(ABLATION_SAVE_DIR,
                        f"{variant_name}_sparsity{sparsity_pct}")


# ─────────────────────────────────────────────────────────────
# EVALUATION — GT METRICS
# Primary metric: GT prob mass (proportion of probability mass on valid YY tokens)
# Secondary metric: GT accuracy
# Patching metric: year_difference head Cohen's d (mirrors S-Inh d in IOI)
# ─────────────────────────────────────────────────────────────

def bootstrap_confidence_interval(
    data: np.ndarray,
    num_bootstrap: int = 1000,
    confidence: float = 0.95
) -> Tuple[float, float]:
    """Compute bootstrap confidence interval."""
    bootstrap_means = []

    for _ in range(num_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(sample.mean())

    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))

    return lower, upper

def test_significance(base_scores: np.ndarray, pruned_scores: np.ndarray,
                     method: str = 'ttest') -> Tuple[float, bool]:
    """Test statistical significance."""
    if len(base_scores) != len(pruned_scores):
        raise ValueError("Base and pruned scores must have same length")

    if method == 'ttest':
        t_stat, p_value = ttest_rel(base_scores, pruned_scores)
    elif method == 'wilcoxon':
        t_stat, p_value = wilcoxon(base_scores, pruned_scores)
    else:
        raise ValueError(f"Unknown method: {method}")

    return float(p_value), p_value < 0.05


def compute_answer_logit(
    logits: torch.Tensor,
    answer_tokens: torch.Tensor,
    end_positions: torch.Tensor
) -> torch.Tensor:
    """
    Compute logit assigned to the correct answer.

    For Greater-Than, we care about P(correct_suffix | prompt).
    Unlike IOI which compares two names, here we just measure
    whether the model assigns high probability to the right answer.
    """
    batch_size = logits.shape[0]

    answer_logits = logits[
        torch.arange(batch_size, device=logits.device),
        end_positions,
        answer_tokens
    ]

    return answer_logits
def cohens_d(base: np.ndarray, pruned: np.ndarray, paired: bool = True) -> float:
    """Calculate Cohen's d effect size."""
    if paired:
        differences = base - pruned
        if differences.std() == 0:
            return 0.0
        return differences.mean() / differences.std()
    else:
        mean_diff = base.mean() - pruned.mean()
        pooled_std = np.sqrt((base.std()**2 + pruned.std()**2) / 2)
        if pooled_std == 0:
            return 0.0
        return mean_diff / pooled_std

def get_yy_token_mapping(tokenizer):
    """Map YY values (00-99) to token IDs (no leading space)."""
    yy_to_token = {}
    for yy in range(100):
        yy_str = f"{yy:02d}"
        tokens = tokenizer(yy_str, return_tensors="pt", add_special_tokens=False)["input_ids"]

        if tokens.shape[1] == 1:
            yy_to_token[yy] = tokens[0, 0].item()
        elif tokens.shape[1] == 2:
            yy_to_token[yy] = tokens[0, -1].item()
        else:
            yy_to_token[yy] = tokens[0, 0].item()

    return yy_to_token


def evaluate_greater_than_performance(
    model: HookedTransformer,
    dataset,
    batch_size: int = 50
) -> Dict[str, float]:
    """
    Evaluate model performance on Greater-Than task using AUTHENTIC paper methodology.

    The paper measures whether the model assigns higher probability to ALL valid
    years (YY > current_YY) vs invalid years (YY <= current_YY).

    Metrics:
    - mean_answer_logit: Average logit for representative correct answer
    - accuracy: Proportion where argmax is a valid YY
    - gt_prob_mass: Average probability mass on ALL valid years (AUTHENTIC METRIC)
    - mean_valid_logit: Average logit across all valid YY tokens
    """
    model.eval()
    device = model.cfg.device

    # Get YY→token mapping
    yy_to_token = get_yy_token_mapping(model.tokenizer)

    all_answer_logits = []
    all_accuracies = []
    all_gt_prob_masses = []  # Probability mass on valid years
    all_mean_valid_logits = []  # Mean logit of valid years

    num_batches = (len(dataset.toks) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in range(num_batches):
            start = i * batch_size
            end = min(start + batch_size, len(dataset.toks))

            batch_tokens = dataset.toks[start:end].to(device)
            batch_answers = dataset.answer_tokenIDs[start:end].to(device)
            batch_ends = dataset.word_idx["end"][start:end].to(device)
            batch_yy_values = dataset.years_YY[start:end].to(device)

            logits = model(batch_tokens)

            # Get logits at prediction position
            batch_logits_at_end = logits[
                torch.arange(len(batch_tokens), device=device),
                batch_ends,
                :
            ]  # Shape: (batch, vocab_size)

            # Compute probabilities
            probs = torch.softmax(batch_logits_at_end, dim=-1)

            # For each example, compute probability mass on valid YY values
            for j in range(len(batch_tokens)):
                current_yy = batch_yy_values[j].item()

                # Get token IDs for all valid YY values (YY > current_yy)
                valid_yy_tokens = [yy_to_token[yy] for yy in range(current_yy + 1, 100)]

                # Sum probability mass over all valid tokens
                valid_token_ids = torch.tensor(valid_yy_tokens, device=device)
                gt_prob_mass = probs[j, valid_token_ids].sum()
                all_gt_prob_masses.append(gt_prob_mass.cpu())

                # Mean logit of valid years
                valid_logits = batch_logits_at_end[j, valid_token_ids]
                mean_valid_logit = valid_logits.mean()
                all_mean_valid_logits.append(mean_valid_logit.cpu())

            # Get logit for representative answer (for backward compatibility)
            answer_logits = compute_answer_logit(
                logits=logits,
                answer_tokens=batch_answers,
                end_positions=batch_ends
            )

            # Compute accuracy: is the argmax a valid year?
            predicted_tokens = batch_logits_at_end.argmax(dim=-1)

            # Check if predicted token is in the valid set
            accuracies = []
            for j in range(len(batch_tokens)):
                current_yy = batch_yy_values[j].item()
                valid_yy_tokens = [yy_to_token[yy] for yy in range(current_yy + 1, 100)]
                is_valid = predicted_tokens[j].item() in valid_yy_tokens
                accuracies.append(float(is_valid))

            all_answer_logits.append(answer_logits.cpu())
            all_accuracies.extend(accuracies)

    all_answer_logits = torch.cat(all_answer_logits, dim=0)
    all_accuracies = torch.tensor(all_accuracies)
    all_gt_prob_masses = torch.stack(all_gt_prob_masses)
    all_mean_valid_logits = torch.stack(all_mean_valid_logits)

    return {
        "mean_answer_logit": float(all_answer_logits.mean()),
        "std_answer_logit": float(all_answer_logits.std()),
        "median_answer_logit": float(all_answer_logits.median()),
        "accuracy": float(all_accuracies.mean()),  # Now truly correct!
        "gt_prob_mass": float(all_gt_prob_masses.mean()),  # ⭐ AUTHENTIC METRIC
        "gt_prob_mass_std": float(all_gt_prob_masses.std()),
        "mean_valid_logit": float(all_mean_valid_logits.mean()),  # Mean logit over valid YY
    }


def activation_patch_head_gt(
    model: HookedTransformer,
    clean_dataset,
    corrupt_dataset,
    layer: int,
    head: int,
    batch_size: int = 50,
    return_samples: bool = True
) -> Union[float, np.ndarray]:
    """
    Activation patching for Greater-Than: patch head output from clean to corrupt.

    Measures: How much does this head's activation matter for getting the right answer?
    """
    model.eval()
    device = model.cfg.device

    hook_name = get_act_name("result", layer)

    all_answer_logits = []
    num_batches = (len(clean_dataset.toks) + batch_size - 1) // batch_size

    with torch.no_grad():
        for i in range(num_batches):
            start = i * batch_size
            end = min(start + batch_size, len(clean_dataset.toks))

            clean_tokens = clean_dataset.toks[start:end].to(device)
            corrupt_tokens = corrupt_dataset.toks[start:end].to(device)

            batch_answers = clean_dataset.answer_tokenIDs[start:end].to(device)
            batch_ends = clean_dataset.word_idx["end"][start:end].to(device)

            # Get clean head activation
            _, clean_cache = model.run_with_cache(clean_tokens)
            clean_head_act = clean_cache[hook_name][:, :, head, :].clone()

            # Patch hook
            def patch_hook(activation, hook):
                activation = activation.clone()
                activation[:, :, head, :] = clean_head_act
                return activation

            patched_logits = model.run_with_hooks(
                corrupt_tokens,
                fwd_hooks=[(hook_name, patch_hook)]
            )

            answer_logits = compute_answer_logit(
                logits=patched_logits,
                answer_tokens=batch_answers,
                end_positions=batch_ends
            )

            all_answer_logits.append(answer_logits.cpu())

    all_answer_logits = torch.cat(all_answer_logits, dim=0)

    if return_samples:
        return all_answer_logits.numpy()
    else:
        return float(all_answer_logits.mean())

def compute_statistical_summary(base_scores, pruned_scores, name="comparison", paired=True):
    """Compute comprehensive statistical summary."""
    base_scores = np.asarray(base_scores).flatten()
    pruned_scores = np.asarray(pruned_scores).flatten()

    if len(base_scores) != len(pruned_scores):
        raise ValueError(f"Base and pruned must have same length")

    base_mean, base_std = base_scores.mean(), base_scores.std()
    pruned_mean, pruned_std = pruned_scores.mean(), pruned_scores.std()

    effect_size = cohens_d(base_scores, pruned_scores, paired=paired)

    p_ttest, sig_ttest = test_significance(base_scores, pruned_scores, method='ttest')
    p_wilcoxon, sig_wilcoxon = test_significance(base_scores, pruned_scores, method='wilcoxon')

    base_ci = bootstrap_confidence_interval(base_scores)
    pruned_ci = bootstrap_confidence_interval(pruned_scores)

    degradation = (base_mean - pruned_mean) / base_mean if base_mean != 0 else 0

    return {
        'name': name,
        'n': len(base_scores),
        'base_mean': float(base_mean),
        'base_std': float(base_std),
        'base_ci_lower': float(base_ci[0]),
        'base_ci_upper': float(base_ci[1]),
        'pruned_mean': float(pruned_mean),
        'pruned_std': float(pruned_std),
        'pruned_ci_lower': float(pruned_ci[0]),
        'pruned_ci_upper': float(pruned_ci[1]),
        'degradation': float(degradation),
        'effect_size_cohens_d': float(effect_size),
        'p_value_ttest': float(p_ttest),
        'significant_ttest': bool(sig_ttest),
        'p_value_wilcoxon': float(p_wilcoxon),
        'significant_wilcoxon': bool(sig_wilcoxon)
    }

def compute_gt_metrics(model_path, gt_clean, gt_corrupt, device, tokenizer,
                       n_samples=300):
    tl = load_model_with_tl(model_path, device, tokenizer)
    tl.cfg.use_attn_result = True

    # Use your existing evaluate_greater_than_performance
    perf = evaluate_greater_than_performance(tl, gt_clean, batch_size=50)

    # Year-difference head patching using your activation_patch_head_gt
    base_scores = activation_patch_head_gt(
        tl, gt_clean, gt_clean,   # clean vs clean = baseline
        layer=9, head=1,
        batch_size=50, return_samples=True)

    patched_scores = activation_patch_head_gt(
        tl, gt_clean, gt_corrupt,  # patch corrupt into clean
        layer=9, head=1,
        batch_size=50, return_samples=True)

    stats = compute_statistical_summary(base_scores, patched_scores,
                                        name="year_diff_L9H1")

    tl.cpu()
    del tl
    torch.cuda.empty_cache()

    return {
        "gt_prob_mass":       perf["gt_prob_mass"],
        "gt_accuracy":        perf["accuracy"],
        "year_diff_cohens_d": stats["effect_size_cohens_d"],
    }

def compute_perplexity(model, tokenizer, texts, device,
                       max_length=512, batch_size=8):
    model.eval()
    total_loss = total_tokens = 0
    enc  = tokenizer(texts, return_tensors="pt", truncation=True,
                     max_length=max_length, padding=True)
    ids, mask = enc["input_ids"], enc["attention_mask"]
    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            b_ids  = ids[i:i+batch_size].to(device)
            b_mask = mask[i:i+batch_size].to(device)
            lbl    = b_ids.clone()
            lbl[b_mask == 0] = -100
            out    = model(input_ids=b_ids, attention_mask=b_mask, labels=lbl)
            n      = (lbl != -100).sum().item()
            total_loss   += out.loss.item() * n
            total_tokens += n
    return float(torch.exp(torch.tensor(total_loss / total_tokens)))


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def run_gt_ablation_study(gt_clean, gt_corrupt, tinystories_texts):
    """
    Main entry point. Call after creating GT datasets:

        tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
        gt_clean, gt_corrupt = create_greater_than_datasets(
            model=tl_model, num_samples=500, seed=42)
        tl_model.cpu(); del tl_model; torch.cuda.empty_cache()

        ts = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
        tinystories_texts = [ex["text"] for ex in list(ts.take(200))]

        results = run_gt_ablation_study(gt_clean, gt_corrupt, tinystories_texts)
    """

    for d in [ABLATION_SAVE_DIR, CACHE_DIR, RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # ── Base model metrics ────────────────────────────────────
    print("\nEvaluating base model...")
    base_hf  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    base_ppl = compute_perplexity(base_hf, tokenizer, tinystories_texts, DEVICE)
    base_hf.cpu()
    del base_hf
    torch.cuda.empty_cache()

    base_gt = compute_gt_metrics(
        os.path.join(EXISTING_MODELS_DIR, "base_model"),
        gt_clean, gt_corrupt, DEVICE, tokenizer)

    print(f"  Base PPL:          {base_ppl:.2f}")
    print(f"  Base GT Prob Mass: {base_gt['gt_prob_mass']:.4f}")
    print(f"  Base GT Accuracy:  {base_gt['gt_accuracy']*100:.1f}%")
    print(f"  Base YD Cohen's d: {base_gt['year_diff_cohens_d']:.3f}")

    # ── Importance scores on GT prompts ───────────────────────
    print("\nComputing importance scores on GT prompts...")
    base_hf = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)

    gt_wrapper = GTDatasetWrapper(gt_clean, tokenizer)
    gt_loader  = DataLoader(gt_wrapper, batch_size=8, shuffle=False)

    layer_scores_heads = compute_layer_gradient_importance(
        base_hf, gt_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_heads_gt.pt"))

    layer_scores_neurons = compute_layer_gradient_importance_neurons(
        base_hf, gt_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_neurons_gt.pt"))

    base_hf.cpu()
    del base_hf
    torch.cuda.empty_cache()

    # ── Diagnostic: does GT scoring implicitly protect GT heads? ──
    run_gt_head_diagnostic(layer_scores_heads)

    # ── Variants ──────────────────────────────────────────────
    variants = [
        # Attribution variants — GT reference data, GT circuit protection
        dict(name="attribution_attention_heads_gt",
             label="Attribution GT – full (attn heads)",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=True,
             use_protected_heads=True),
        dict(name="ablation_gt_no_budgets",
             label="Attribution GT – no layer budgets (attn)",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=False,
             use_protected_heads=True),
        dict(name="ablation_gt_no_protection",
             label="Attribution GT – no head protection",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=True,
             use_protected_heads=False),
        dict(name="ablation_gt_no_budgets_no_protection",
             label="Attribution GT – no budgets no protection",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=False,
             use_protected_heads=False),

        # Neuron variants
        dict(name="attribution_neurons_gt",
             label="Attribution GT – full (neurons)",
             load_existing=False,
             component="neurons",
             use_layer_budgets=True),
        dict(name="ablation_gt_neurons_no_budgets",
             label="Attribution GT – no layer budgets (neurons)",
             load_existing=False,
             component="neurons",
             use_layer_budgets=False),

        # Baselines — same saved models as IOI ablation
        dict(name="magnitude_attention_heads",
             label="Magnitude – attn heads",
             load_existing=True,
             component="attention_heads"),
        dict(name="magnitude_neurons",
             label="Magnitude – neurons",
             load_existing=True,
             component="neurons"),
        dict(name="random_attention_heads",
             label="Random – attn heads",
             load_existing=True,
             component="attention_heads"),
        dict(name="random_neurons",
             label="Random – neurons",
             load_existing=True,
             component="neurons"),
    ]

    all_results = []

    for v in variants:
        print(f"\n{'='*60}")
        print(f"Variant: {v['label']}")
        print(f"{'='*60}")

        for sparsity_pct in SPARSITY_LEVELS:
            model_path = get_model_path(v["name"], sparsity_pct)
            print(f"  Path:   {model_path}")
            print(f"  Exists: {os.path.exists(model_path)}")

            # Build new models
            if not v["load_existing"] and not os.path.exists(model_path):
                print(f"\n  Pruning sparsity {sparsity_pct}%...")
                base_fresh = GPT2LMHeadModel.from_pretrained("gpt2")

                if v["component"] == "attention_heads":
                    pruned = prune_attention_heads_ablation(
                        base_fresh, layer_scores_heads, sparsity_pct,
                        use_layer_budgets=v["use_layer_budgets"],
                        use_protected_heads=v["use_protected_heads"],
                    )
                else:
                    pruned = prune_neurons_ablation(
                        base_fresh, layer_scores_neurons, sparsity_pct,
                        use_layer_budgets=v["use_layer_budgets"],
                    )

                os.makedirs(model_path, exist_ok=True)
                pruned.save_pretrained(model_path)
                tokenizer.save_pretrained(model_path)
                del base_fresh, pruned
                torch.cuda.empty_cache()
                print(f"    Saved to {model_path}")

            if not os.path.exists(model_path):
                print(f"  ⚠️  Not found: {model_path} — skipping")
                continue

            print(f"\n  Evaluating sparsity {sparsity_pct}%...")

            # Perplexity
            hf_model = GPT2LMHeadModel.from_pretrained(model_path).to(DEVICE)
            ppl      = compute_perplexity(hf_model, tokenizer,
                                          tinystories_texts, DEVICE)
            hf_model.cpu()
            del hf_model
            torch.cuda.empty_cache()

            actual = get_actual_sparsity(model_path)
            gt     = compute_gt_metrics(model_path, gt_clean, gt_corrupt,
                                        DEVICE, tokenizer)

            row = {
                "variant":              v["label"],
                "nominal_sparsity":     sparsity_pct,
                "actual_sparsity_pct":  round(actual * 100, 1),
                "perplexity":           round(ppl, 2),
                "ppl_delta_pct":        round((ppl - base_ppl) / base_ppl * 100, 1),
                "gt_prob_mass":         gt["gt_prob_mass"],
                "gt_accuracy_pct":      round(gt["gt_accuracy"] * 100, 1),
                "year_diff_cohens_d":   gt["year_diff_cohens_d"],
            }
            all_results.append(row)

            print(f"    Actual sparsity:   {actual*100:.1f}%")
            print(f"    Perplexity:        {ppl:.1f} (+{row['ppl_delta_pct']}%)")
            print(f"    GT Prob Mass:      {gt['gt_prob_mass']:.4f}")
            print(f"    GT Accuracy:       {gt['gt_accuracy']*100:.1f}%")
            print(f"    YD Cohen's d:      {gt['year_diff_cohens_d']:.3f}")

    # ── Save ──────────────────────────────────────────────────
    os.makedirs(RESULTS_DIR, exist_ok=True)
    csv_path = os.path.join(RESULTS_DIR, "ablation_results_gt.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)
    print(f"\n✓ Saved to {csv_path}")

    with open(os.path.join(RESULTS_DIR, "base_metrics_gt.json"), "w") as f:
        json.dump({
            "ppl":              base_ppl,
            "gt_prob_mass":     base_gt["gt_prob_mass"],
            "gt_accuracy":      base_gt["gt_accuracy"],
            "year_diff_d":      base_gt["year_diff_cohens_d"],
        }, f, indent=2)

    print_table(all_results, base_ppl, base_gt)
    return all_results


# ─────────────────────────────────────────────────────────────
# TABLE
# ─────────────────────────────────────────────────────────────

def print_table(results, base_ppl, base_gt):
    print(f"\n{'='*100}")
    print("GT ABLATION STUDY RESULTS")
    print(f"Base: PPL={base_ppl:.1f} | GT Prob Mass={base_gt['gt_prob_mass']:.4f} | "
          f"GT Acc={base_gt['gt_accuracy']*100:.1f}% | "
          f"YD d={base_gt['year_diff_cohens_d']:.3f}")
    print(f"{'='*100}")

    for sparsity_pct in SPARSITY_LEVELS:
        rows = [r for r in results if r["nominal_sparsity"] == sparsity_pct]
        print(f"\n── {sparsity_pct}% Nominal Sparsity ──")
        print(f"{'Variant':<42} {'Act%':>5} {'PPL':>8} {'PPLΔ%':>7} "
              f"{'GT Mass':>8} {'GT Acc':>8} {'YD d':>8}")
        print("-" * 92)
        for r in rows:
            print(f"{r['variant']:<42} "
                  f"{r['actual_sparsity_pct']:>5.1f} "
                  f"{r['perplexity']:>8.1f} "
                  f"{r['ppl_delta_pct']:>6.0f}% "
                  f"{r['gt_prob_mass']:>8.4f} "
                  f"{r['gt_accuracy_pct']:>7.1f}% "
                  f"{r['year_diff_cohens_d']:>8.3f}")


# ─────────────────────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────────────────────
#
# In your Colab notebook:
#
# from transformer_lens import HookedTransformer
# from datasets import load_dataset
#
# # 1. Create GT datasets (needs TL model for tokenization)
# tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
# gt_clean, gt_corrupt = create_greater_than_datasets(
#     model=tl_model, num_samples=500, seed=42)
# tl_model.cpu(); del tl_model; torch.cuda.empty_cache()
#
# # 2. Load TinyStories for perplexity
# ts = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
# tinystories_texts = [ex["text"] for ex in list(ts.take(200))]
#
# # 3. Run GT ablation
# results = run_gt_ablation_study(gt_clean, gt_corrupt, tinystories_texts)

In [ ]:

def run_gt_diagnostic(gt_clean, gt_corrupt, tinystories_texts,
                      layer_scores_heads, layer_scores_neurons):
    """
    Quick sanity check before running full GT ablation.

    Args:
        gt_clean, gt_corrupt: from create_greater_than_datasets()
        tinystories_texts: list of strings for perplexity
        layer_scores_heads: from compute_layer_gradient_importance() on GT prompts
        layer_scores_neurons: from compute_layer_gradient_importance_neurons() on GT prompts
    """

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    print("=" * 60)
    print("GT DIAGNOSTIC CHECK")
    print("=" * 60)

    # ── [1/5] Importance score sanity check ──────────────────
    print("\n[1/5] GT importance score sanity check...")

    for layer_idx in [5, 7, 9, 11]:
        scores  = layer_scores_heads[layer_idx]
        ranked  = sorted(range(N_HEADS), key=lambda h: scores[h].item())
        circuit = GT_PROTECTED_HEADS.get(layer_idx, [])
        budget  = LAYER_BUDGETS_ATTN[layer_idx]
        nominal = 0.30

        MAX_HEADS_PER_LAYER = {
            0: 2, 1: 2, 2: 2, 3: 2,
            4: 3, 5: 3, 6: 3, 7: 3,
            8: 2, 9: 1, 10: 1, 11: 1,
        }

        eff     = nominal * budget
        n_prune = min(
            max(0, int(N_HEADS * eff)),
            MAX_HEADS_PER_LAYER[layer_idx],
            int(N_HEADS * 0.40),
        )
        would_prune = [h for h in ranked[:n_prune] if h not in set(circuit)]

        print(f"  Layer {layer_idx} head scores — "
              f"min: {scores.min():.4f}, max: {scores.max():.4f}, "
              f"mean: {scores.mean():.4f}")
        print(f"         budget={budget}, n_prune={n_prune}, "
              f"gt_circuit={circuit}, would_prune={would_prune}")

    # ── [2/5] GT circuit head diagnostic ─────────────────────
    print("\n[2/5] GT circuit head diagnostic...")
    print("  Do GT importance scores implicitly protect GT circuit heads?")
    print("  (rank 11 = most important, rank 0 = pruned first)")

    all_protected = True
    for layer_idx, heads in GT_PROTECTED_HEADS.items():
        scores  = layer_scores_heads[layer_idx]
        ranked  = sorted(range(N_HEADS), key=lambda h: scores[h].item())
        ranks   = [ranked.index(h) for h in heads]

        # A head is implicitly protected if it ranks in top 4 (positions 8-11)
        implicitly_safe = [r >= 8 for r in ranks]
        status = "✓ IMPLICIT" if all(implicitly_safe) else "⚠ AT RISK"

        if not all(implicitly_safe):
            all_protected = False

        print(f"  Layer {layer_idx}: GT heads {heads} "
              f"at rank positions {ranks} — {status}")

    if all_protected:
        print("\n  ✓ All GT circuit heads naturally rank highly on GT importance scores")
        print("    → Explicit protection redundant (same finding as IOI)")
    else:
        print("\n  ⚠ Some GT circuit heads rank low — explicit protection IS needed")
        print("    → Different from IOI finding — worth noting in dissertation")

    # ── [3/5] Test pruning at 30% ─────────────────────────────
    print("\n[3/5] Test pruning at 30% sparsity...")

    base_fresh = GPT2LMHeadModel.from_pretrained("gpt2")
    test_pruned = prune_attention_heads_ablation(
        base_fresh, layer_scores_heads, 30,
        use_layer_budgets=True,
        use_protected_heads=True,
    )

    print("  Per-layer attention sparsity:")
    total_pruned = 0
    for layer_idx in range(N_LAYERS):
        attn   = test_pruned.transformer.h[layer_idx].attn
        w      = attn.c_proj.weight  # [N_HEADS*HEAD_DIM, hidden]
        zeroed = 0
        for h in range(N_HEADS):
            s = h * HEAD_DIM
            e = s + HEAD_DIM
            if (w[s:e, :] == 0).all():
                zeroed += 1
        total_pruned += zeroed
        prot = GT_PROTECTED_HEADS.get(layer_idx, [])
        prot_str = f" (GT circuit heads: {prot})" if prot else ""
        print(f"    Layer {layer_idx}: {zeroed/N_HEADS*100:.1f}% sparse{prot_str}")

    overall = total_pruned / (N_LAYERS * N_HEADS)
    print(f"  Overall attention sparsity: {overall*100:.1f}%")

    # ── [4/5] Quick performance check ────────────────────────
    print("\n[4/5] Quick performance check...")

    # Save test model temporarily
    import tempfile, shutil
    tmp_dir = tempfile.mkdtemp()
    test_pruned.save_pretrained(tmp_dir)
    tokenizer.save_pretrained(tmp_dir)

    # Perplexity on subset
    hf_test = GPT2LMHeadModel.from_pretrained(tmp_dir).to(DEVICE)
    ppl = compute_perplexity(hf_test, tokenizer, tinystories_texts[:20], DEVICE)
    hf_test.cpu()
    del hf_test
    torch.cuda.empty_cache()

    print(f"  Test model perplexity: {ppl:.2f}")
    print(f"  (Base is ~10 — if >80 something is wrong)")

    # GT metrics on small sample
    gt_metrics = compute_gt_metrics(
        tmp_dir, gt_clean, gt_corrupt, DEVICE, tokenizer,
        n_samples=50)

    shutil.rmtree(tmp_dir)
    del base_fresh, test_pruned
    torch.cuda.empty_cache()

    print(f"  Test GT prob mass:   {gt_metrics['gt_prob_mass']:.4f}")
    print(f"  Test GT accuracy:    {gt_metrics['gt_accuracy']*100:.1f}%")
    print(f"  Test YD Cohen's d:   {gt_metrics['year_diff_cohens_d']:.3f}")
    print(f"  (Base GT prob mass ~0.855, accuracy ~94% — if acc <40% something is wrong)")

    # ── [5/5] Neuron score sanity check ──────────────────────
    print("\n[5/5] Neuron score sanity check...")
    for layer_idx in [0, 5, 9, 11]:
        scores = layer_scores_neurons[layer_idx]
        nonzero = (scores > 0).sum().item()
        print(f"  Layer {layer_idx} neuron scores — "
              f"min: {scores.min():.4f}, max: {scores.max():.4f}, "
              f"mean: {scores.mean():.4f}, nonzero: {nonzero}/{len(scores)}")

    # ── Summary ───────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("GT DIAGNOSTIC SUMMARY")
    print("=" * 60)

    ppl_ok = ppl < 80
    acc_ok = gt_metrics['gt_accuracy'] > 0.40
    sparsity_ok = 0.03 <= overall <= 0.15

    print(f"  Perplexity:     {ppl:.1f}  {'✓' if ppl_ok else '✗ TOO HIGH'}")
    print(f"  GT Accuracy:    {gt_metrics['gt_accuracy']*100:.1f}%  {'✓' if acc_ok else '✗ TOO LOW'}")
    print(f"  GT Prob Mass:   {gt_metrics['gt_prob_mass']:.4f}")
    print(f"  Sparsity:       {overall*100:.1f}%  {'✓' if sparsity_ok else '✗ CHECK BUDGETS'}")

    if ppl_ok and acc_ok and sparsity_ok:
        print("\n  ✓ All checks passed — safe to run full GT ablation")
    else:
        print("\n  ✗ Issues found — do NOT run full ablation until resolved")

    return {
        "ppl": ppl,
        "gt_prob_mass": gt_metrics["gt_prob_mass"],
        "gt_accuracy": gt_metrics["gt_accuracy"],
        "year_diff_cohens_d": gt_metrics["year_diff_cohens_d"],
        "overall_sparsity": overall,
        "all_checks_passed": ppl_ok and acc_ok and sparsity_ok,
    }


# ─────────────────────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────────────────────

# Step 1 — Create GT datasets
from transformer_lens import HookedTransformer
tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
gt_clean, gt_corrupt = create_greater_than_datasets(
    model=tl_model, num_samples=500, seed=42)
tl_model.cpu(); del tl_model; torch.cuda.empty_cache()

# Step 2 — Compute GT importance scores
from transformers import GPT2LMHeadModel, GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

base_hf = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
gt_wrapper = GTDatasetWrapper(gt_clean, tokenizer)
gt_loader  = DataLoader(gt_wrapper, batch_size=8, shuffle=False)

layer_scores_heads = compute_layer_gradient_importance(
    base_hf, gt_loader, DEVICE,
    cache_path=os.path.join(CACHE_DIR, "grad_importance_heads_gt.pt"))
layer_scores_neurons = compute_layer_gradient_importance_neurons(
    base_hf, gt_loader, DEVICE,
    cache_path=os.path.join(CACHE_DIR, "grad_importance_neurons_gt.pt"))
base_hf.cpu(); del base_hf; torch.cuda.empty_cache()

# Step 3 — Load TinyStories
from datasets import load_dataset
ts = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
tinystories_texts = [ex["text"] for ex in list(ts.take(200))]

# Step 4 — Run diagnostic
diag = run_gt_diagnostic(
    gt_clean, gt_corrupt, tinystories_texts,
    layer_scores_heads, layer_scores_neurons
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained model gpt2 into HookedTransformer

[VALIDATION] Greater-Than Dataset (Hanna et al. 2023 authentic):
  Generated 500 examples
  Year range: 1000-1999
  Balanced: True

  Example 1:
    Year: 1902 (XX=19, YY=2)
    Noun: movement
    Clean: The movement lasted from the year 1902 to the year 19
    Corrupt: The movement lasted from the year 1901 to the year 19
    Tokens: <|endoftext|>The movement lasted from the year 1902 to the year 19<|endoftext|>
    End position: 11
    Token at end: ' 19'
    Next token: '<|endoftext|>'
    Valid YY range: 3 to 99
    Representative answer token: 1828
    Decodes to: '22'

  Example 2:
    Year: 1803 (XX=18, YY=3)
    Noun: war
    Clean: The war lasted from the year 1803 to the year 18
    Corrupt: The war lasted from the year 1801 to the year 18
    Tokens: <|endoftext|>The war lasted from the year 1803 to the year 18
    End position: 12
    Token at end: ' 18'
    Valid YY range: 4 to 99
    Representative answer token: 1954
  

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Test model perplexity: 60.26
  (Base is ~10 — if >80 something is wrong)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  MLP sparsity — HF: 0.00% | TL: 0.00% ✓
Moving model to device:  cpu
  Test GT prob mass:   0.8868
  Test GT accuracy:    96.6%
  Test YD Cohen's d:   0.499
  (Base GT prob mass ~0.855, accuracy ~94% — if acc <40% something is wrong)

[5/5] Neuron score sanity check...
  Layer 0 neuron scores — min: 0.0002, max: 0.0263, mean: 0.0023, nonzero: 3072/3072
  Layer 5 neuron scores — min: 0.0005, max: 0.0186, mean: 0.0024, nonzero: 3072/3072
  Layer 9 neuron scores — min: 0.0003, max: 0.0514, mean: 0.0019, nonzero: 3072/3072
  Layer 11 neuron scores — min: 0.0003, max: 0.0494, mean: 0.0018, nonzero: 3072/3072

GT DIAGNOSTIC SUMMARY
  Perplexity:     60.3  ✓
  GT Accuracy:    96.6%  ✓
  GT Prob Mass:   0.8868
  Sparsity:       9.0%  ✓

  ✓ All checks passed — safe to run full GT ablation


In [ ]:
from transformer_lens import HookedTransformer
from datasets import load_dataset

tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
gt_clean, gt_corrupt = create_greater_than_datasets(
    model=tl_model, num_samples=500, seed=42)
tl_model.cpu(); del tl_model; torch.cuda.empty_cache()

ts = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
tinystories_texts = [ex["text"] for ex in list(ts.take(200))]

results = run_gt_ablation_study(gt_clean, gt_corrupt, tinystories_texts)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained model gpt2 into HookedTransformer

[VALIDATION] Greater-Than Dataset (Hanna et al. 2023 authentic):
  Generated 500 examples
  Year range: 1000-1999
  Balanced: True

  Example 1:
    Year: 1902 (XX=19, YY=2)
    Noun: movement
    Clean: The movement lasted from the year 1902 to the year 19
    Corrupt: The movement lasted from the year 1901 to the year 19
    Tokens: <|endoftext|>The movement lasted from the year 1902 to the year 19<|endoftext|>
    End position: 11
    Token at end: ' 19'
    Next token: '<|endoftext|>'
    Valid YY range: 3 to 99
    Representative answer token: 1828
    Decodes to: '22'

  Example 2:
    Year: 1803 (XX=18, YY=3)
    Noun: war
    Clean: The war lasted from the year 1803 to the year 18
    Corrupt: The war lasted from the year 1801 to the year 18
    Tokens: <|endoftext|>The war lasted from the year 1803 to the year 18
    End position: 12
    Token at end: ' 18'
    Valid YY range: 4 to 99
    Representative answer token: 1954
  

README.md: 0.00B [00:00, ?B/s]


Evaluating base model...


...

  MLP sparsity — HF: 50.00% | TL: 50.00% ✓
Moving model to device:  cpu
    Actual sparsity:   22.8%
    Perplexity:        275.0 (+2645.7%)
    GT Prob Mass:      0.0335
    GT Accuracy:       0.0%
    YD Cohen's d:      -1.029
  Path:   /content/drive/MyDrive/IRP/GPT-2/pruned_models/random_neurons_sparsity70
  Exists: True

  Evaluating sparsity 70%...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  MLP sparsity — HF: 69.99% | TL: 69.99% ✓
Moving model to device:  cpu
    Actual sparsity:   31.9%
    Perplexity:        1095.6 (+10839.2%)
    GT Prob Mass:      0.0003
    GT Accuracy:       0.0%
    YD Cohen's d:      -0.164

✓ Saved to /content/drive/MyDrive/IRP/ablation_results_gt/ablation_results_gt.csv

GT ABLATION STUDY RESULTS
Base: PPL=10.0 | GT Prob Mass=0.8616 | GT Acc=94.6% | YD d=0.684

── 30% Nominal Sparsity ──
Variant                                     Act%      PPL   PPLΔ%  GT Mass   GT Acc     YD d
--------------------------------------------------------------------------------------------
Attribution GT – full (attn heads)           2.1     46.1    361%   0.8868    96.6%    0.499
Attribution GT – no layer budgets (attn)     5.7    157.2   1469%   0.6969    76.8%   -0.741
Attribution GT – no head protection          2.1     46.1    361%   0.8868    96.6%    0.499
Attribution GT – no budgets no protection    5.7    153.7   1434%   0.7013    84.4%   -0.442
Attribut

#Ablation Tiny Stories


In [ ]:
!pip install transformers transformer-lens torch scikit-learn matplotlib seaborn pandas tqdm

INFO: pip is looking at multiple versions of transformer-lens to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.0 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=ed25a2431768302b9180535994890235eb8c6c805fdd5b23627d72931d1c38c2
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator
  Attempting uninstall: numpy
    Found existing instal

In [ ]:
"""
Ablation Study — TinyStories (General Language Modelling)
==========================================================
Mirrors the GT and IOI ablation studies exactly but uses:
- TinyStories for importance score computation
- Perplexity as the primary evaluation metric
- Token-level next-word prediction accuracy as secondary metric
- No circuit-specific protected heads (general language modelling task)
- Same pruning logic, budgets, and variant structure as GT/IOI v5

This script answers: does attribution-guided pruning preserve general
language modelling capability better than magnitude/random pruning at
equivalent sparsity?
"""

import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.loading_from_pretrained import convert_gpt2_weights
import numpy as np
import json, os, csv
from tqdm import tqdm
from copy import deepcopy
from torch.utils.data import Dataset, DataLoader
from scipy.stats import ttest_rel, wilcoxon
from typing import Tuple, Dict, List


# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EXISTING_MODELS_DIR = "/content/drive/MyDrive/IRP/GPT-2/pruned_models"
ABLATION_SAVE_DIR   = "/content/drive/MyDrive/IRP/ablation_variants_ts"
CACHE_DIR           = "/content/drive/MyDrive/IRP/ablation_cache_ts"
RESULTS_DIR         = "/content/drive/MyDrive/IRP/ablation_results_ts"

SPARSITY_LEVELS = [30, 50, 70]

# No task-specific protected heads for TinyStories —
# importance scoring alone drives pruning decisions.
# Set to empty dict; flip use_protected_heads=True with this to
# test the effect of adding protection later.
TS_PROTECTED_HEADS: Dict[int, List[int]] = {}

# Layer budgets — identical to IOI/GT scripts (CKA findings are circuit-agnostic)
LAYER_BUDGETS_ATTN = {
    0: 0.4,
    1: 0.4,
    2: 0.4,
    3: 0.4,
    4: 0.4,
    5: 0.4,
    6: 0.4,
    7: 0.4,
    8: 0.6,
    9: 0.5,
    10: 0.4,
    11: 0.3,
}

LAYER_BUDGETS_NEURONS = {
    0: 1.50,
    1: 1.50,
    2: 1.40,
    3: 1.30,
    4: 1.10,
    5: 1.00,
    6: 0.90,
    7: 0.85,
    8: 0.75,
    9: 0.55,
    10: 0.50,
    11: 0.35,
}

N_LAYERS    = 12
N_HEADS     = 12
HIDDEN_SIZE = 768
HEAD_DIM    = HIDDEN_SIZE // N_HEADS  # 64
MLP_SIZE    = 3072

# TinyStories split sizes
N_IMPORTANCE = 500   # prompts used to compute gradient importance scores
N_EVAL       = 200   # prompts used for perplexity / accuracy evaluation
MAX_LENGTH   = 256   # token truncation length


# ─────────────────────────────────────────────────────────────
# WEIGHT LOADING
# ─────────────────────────────────────────────────────────────

def load_model_with_tl(model_path, device, tokenizer=None):
    hf_model = GPT2LMHeadModel.from_pretrained(model_path)

    cfg = HookedTransformerConfig(
        n_layers=12, d_model=768, n_heads=12, d_head=64,
        d_mlp=3072, d_vocab=50257, n_ctx=1024,
        act_fn="gelu_new", normalization_type="LN", device=device,
    )

    tl_model = HookedTransformer(cfg)
    tl_state  = convert_gpt2_weights(hf_model, cfg)
    tl_model.load_state_dict(tl_state, strict=False)

    hf_sp = (hf_model.transformer.h[0].mlp.c_fc.weight == 0).float().mean().item()
    tl_sp = (tl_model.blocks[0].mlp.W_in == 0).float().mean().item()
    print(f"  MLP sparsity — HF: {hf_sp:.2%} | TL: {tl_sp:.2%}", end="")
    print(" ✓" if abs(hf_sp - tl_sp) < 0.01 else " ⚠️  mismatch")

    del hf_model
    torch.cuda.empty_cache()
    tl_model.eval()
    if tokenizer is not None:
        tl_model.set_tokenizer(tokenizer)
    return tl_model


def get_actual_sparsity(model_path):
    m = GPT2LMHeadModel.from_pretrained(model_path)
    total = zeros = 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p.data == 0).sum().item()
    del m
    return zeros / total


# ─────────────────────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────────────────────

class TinyStoriesDataset(Dataset):
    """
    Wraps a list of TinyStories text strings for use with a DataLoader.
    Returns input_ids, attention_mask, and labels for causal LM training.
    """
    def __init__(self, texts: List[str], tokenizer, max_length: int = MAX_LENGTH):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            return_tensors="pt",
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
        )
        ids  = enc["input_ids"].squeeze(0)
        mask = enc["attention_mask"].squeeze(0)
        lbl  = ids.clone()
        lbl[mask == 0] = -100          # ignore padding in loss
        return {"input_ids": ids, "attention_mask": mask, "labels": lbl}


def load_tinystories(n_importance: int = N_IMPORTANCE,
                     n_eval: int = N_EVAL,
                     seed: int = 42) -> Tuple[List[str], List[str]]:
    """
    Load TinyStories validation split.
    Returns (importance_texts, eval_texts) — non-overlapping.
    """
    from datasets import load_dataset
    ds = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
    texts = [ex["text"] for ex in list(ds.take(n_importance + n_eval))]

    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(texts))
    imp_texts  = [texts[i] for i in idx[:n_importance]]
    eval_texts = [texts[i] for i in idx[n_importance:n_importance + n_eval]]
    return imp_texts, eval_texts


# ─────────────────────────────────────────────────────────────
# IMPORTANCE SCORING
# ─────────────────────────────────────────────────────────────

def compute_gradient_importance(model, dataloader, device, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        print(f"  Loading cached importance from {cache_path}")
        return torch.load(cache_path, map_location="cpu")

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    print("  Computing gradient × activation importance on TinyStories...")
    model.train()
    importance = {n: torch.zeros_like(p, device="cpu")
                  for n, p in model.named_parameters() if p.requires_grad}

    for batch in tqdm(dataloader, desc="  Gradient batches"):
        model.zero_grad()
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        out.loss.backward()
        for name, param in model.named_parameters():
            if param.grad is not None:
                importance[name] += (param.grad.abs() * param.data.abs()).cpu()

    model.eval()
    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        torch.save(importance, cache_path)
        print(f"  Saved to {cache_path}")
    return importance


def compute_layer_gradient_importance(model, dataloader, device, cache_path=None):
    raw = compute_gradient_importance(model, dataloader, device, cache_path)
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        if key in raw:
            imp = raw[key]
            scores = torch.tensor([
                imp[h * HEAD_DIM:(h + 1) * HEAD_DIM, :].mean().item()
                for h in range(N_HEADS)
            ])
        else:
            scores = torch.zeros(N_HEADS)
        layer_scores[layer_idx] = scores
    return layer_scores


def compute_layer_gradient_importance_neurons(model, dataloader, device, cache_path=None):
    raw = compute_gradient_importance(model, dataloader, device, cache_path)
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.mlp.c_fc.weight"
        if key in raw:
            imp = raw[key]
            scores = imp.mean(dim=0)
        else:
            scores = torch.zeros(MLP_SIZE)
        layer_scores[layer_idx] = scores
    return layer_scores


def run_ts_head_diagnostic(layer_scores_heads):
    """
    Print the top-3 most important heads per layer according to TinyStories
    importance scores. Useful for sanity-checking that induction/name-mover
    heads rank highly (as expected from prior analyses).
    """
    print("\n" + "=" * 60)
    print("TINYSTORIES HEAD IMPORTANCE DIAGNOSTIC")
    print("Top-3 most important heads per layer (TinyStories scoring)")
    print("=" * 60)
    for layer_idx in range(N_LAYERS):
        scores = layer_scores_heads[layer_idx]
        top3 = sorted(range(N_HEADS), key=lambda h: scores[h].item(), reverse=True)[:3]
        vals = [f"H{h}({scores[h]:.3f})" for h in top3]
        print(f"  L{layer_idx:02d}: {', '.join(vals)}")


# ─────────────────────────────────────────────────────────────
# PRUNING
# ─────────────────────────────────────────────────────────────

def _apply_attention_mask(attn, mask):
    with torch.no_grad():
        for i, keep in enumerate(mask):
            if not keep:
                s = i * HEAD_DIM
                e = s + HEAD_DIM
                attn.c_attn.weight[:, s:e] = 0
                attn.c_attn.weight[:, HIDDEN_SIZE + s:HIDDEN_SIZE + e] = 0
                attn.c_attn.weight[:, 2 * HIDDEN_SIZE + s:2 * HIDDEN_SIZE + e] = 0
                attn.c_proj.weight[s:e, :] = 0


def prune_attention_heads_ablation(base_model, importance_scores_per_layer,
                                   sparsity_pct, use_layer_budgets,
                                   use_protected_heads):
    nominal = sparsity_pct / 100
    pruned  = deepcopy(base_model)

    MAX_HEADS_PER_LAYER = {
        0: 2, 1: 2, 2: 2, 3: 2,
        4: 3, 5: 3, 6: 3, 7: 3,
        8: 2, 9: 1, 10: 1, 11: 1,
    }

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget = LAYER_BUDGETS_ATTN[layer_idx] if use_layer_budgets else 1.0
            eff    = nominal * budget

            if use_layer_budgets:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    MAX_HEADS_PER_LAYER[layer_idx],
                    int(N_HEADS * 0.40),
                )
            else:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    int(N_HEADS * 0.70),
                )

            protected = set(TS_PROTECTED_HEADS.get(layer_idx, [])) \
                        if use_protected_heads else set()

            scores = importance_scores_per_layer[layer_idx]
            ranked = sorted(range(N_HEADS), key=lambda h: scores[h].item())

            mask = torch.ones(N_HEADS, dtype=torch.bool)
            pruned_count = 0
            for h in ranked:
                if pruned_count >= n_prune:
                    break
                if h not in protected:
                    mask[h] = False
                    pruned_count += 1

            attn = pruned.transformer.h[layer_idx].attn
            _apply_attention_mask(attn, mask)

    return pruned


def prune_neurons_ablation(base_model, importance_scores_per_layer,
                           sparsity_pct, use_layer_budgets):
    nominal = sparsity_pct / 100
    pruned  = deepcopy(base_model)

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget  = LAYER_BUDGETS_NEURONS[layer_idx] if use_layer_budgets else 1.0
            eff     = min(nominal * budget, 0.85)
            n_prune = max(0, int(MLP_SIZE * eff))

            scores = importance_scores_per_layer[layer_idx]
            ranked = torch.argsort(scores)

            mask = torch.ones(MLP_SIZE, dtype=torch.bool)
            mask[ranked[:n_prune]] = False

            block    = pruned.transformer.h[layer_idx]
            fc_layer = block.mlp.c_fc
            with torch.no_grad():
                fc_layer.weight[:, ~mask] = 0
                if fc_layer.bias is not None:
                    fc_layer.bias[~mask] = 0
                block.mlp.c_proj.weight[~mask, :] = 0

    return pruned


# ─────────────────────────────────────────────────────────────
# PATH HELPERS
# ─────────────────────────────────────────────────────────────

def get_model_path(variant_name, sparsity_pct):
    existing = {
        "magnitude_attention_heads": EXISTING_MODELS_DIR,
        "magnitude_neurons":         EXISTING_MODELS_DIR,
        "random_attention_heads":    EXISTING_MODELS_DIR,
        "random_neurons":            EXISTING_MODELS_DIR,
    }
    if variant_name in existing:
        return os.path.join(existing[variant_name],
                            f"{variant_name}_sparsity{sparsity_pct}")
    return os.path.join(ABLATION_SAVE_DIR,
                        f"{variant_name}_sparsity{sparsity_pct}")


# ─────────────────────────────────────────────────────────────
# EVALUATION — TINYSTORIES METRICS
# Primary metric:   perplexity on held-out TinyStories eval set
# Secondary metric: next-token prediction accuracy
# Both measure general language modelling capability preservation.
# ─────────────────────────────────────────────────────────────

def bootstrap_confidence_interval(data: np.ndarray,
                                   num_bootstrap: int = 1000,
                                   confidence: float = 0.95) -> Tuple[float, float]:
    bootstrap_means = []
    for _ in range(num_bootstrap):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_means.append(sample.mean())
    alpha = 1 - confidence
    lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    return lower, upper


def cohens_d(base: np.ndarray, pruned: np.ndarray, paired: bool = True) -> float:
    if paired:
        differences = base - pruned
        if differences.std() == 0:
            return 0.0
        return float(differences.mean() / differences.std())
    else:
        mean_diff  = base.mean() - pruned.mean()
        pooled_std = np.sqrt((base.std() ** 2 + pruned.std() ** 2) / 2)
        return 0.0 if pooled_std == 0 else float(mean_diff / pooled_std)


def compute_perplexity(model, tokenizer, texts, device,
                       max_length: int = MAX_LENGTH,
                       batch_size: int = 8) -> float:
    """Compute per-token cross-entropy perplexity on a list of texts."""
    model.eval()
    total_loss = total_tokens = 0
    enc  = tokenizer(texts, return_tensors="pt", truncation=True,
                     max_length=max_length, padding=True)
    ids, mask = enc["input_ids"], enc["attention_mask"]
    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            b_ids  = ids[i:i + batch_size].to(device)
            b_mask = mask[i:i + batch_size].to(device)
            lbl    = b_ids.clone()
            lbl[b_mask == 0] = -100
            out    = model(input_ids=b_ids, attention_mask=b_mask, labels=lbl)
            n      = (lbl != -100).sum().item()
            total_loss   += out.loss.item() * n
            total_tokens += n
    return float(torch.exp(torch.tensor(total_loss / total_tokens)))


def compute_next_token_accuracy(model, tokenizer, texts, device,
                                 max_length: int = MAX_LENGTH,
                                 batch_size: int = 8) -> Dict[str, float]:
    """
    Compute next-token prediction accuracy (top-1 and top-5).
    For each non-padding token position, checks whether the argmax
    prediction matches the ground truth.
    """
    model.eval()
    total = top1_correct = top5_correct = 0

    enc  = tokenizer(texts, return_tensors="pt", truncation=True,
                     max_length=max_length, padding=True)
    ids, mask = enc["input_ids"], enc["attention_mask"]

    with torch.no_grad():
        for i in range(0, len(ids), batch_size):
            b_ids  = ids[i:i + batch_size].to(device)
            b_mask = mask[i:i + batch_size].to(device)

            logits = model(input_ids=b_ids).logits  # (B, T, V)

            # Shift: predict token t+1 from position t
            shift_logits = logits[:, :-1, :]         # (B, T-1, V)
            shift_labels = b_ids[:, 1:]               # (B, T-1)
            shift_mask   = b_mask[:, 1:]              # (B, T-1)

            # Top-1 accuracy
            preds_top1 = shift_logits.argmax(dim=-1)  # (B, T-1)
            correct_top1 = (preds_top1 == shift_labels) & (shift_mask == 1)
            top1_correct += correct_top1.sum().item()

            # Top-5 accuracy
            preds_top5 = shift_logits.topk(5, dim=-1).indices  # (B, T-1, 5)
            correct_top5 = (preds_top5 == shift_labels.unsqueeze(-1)).any(-1) \
                           & (shift_mask == 1)
            top5_correct += correct_top5.sum().item()

            total += shift_mask.sum().item()

    return {
        "top1_accuracy": top1_correct / total if total > 0 else 0.0,
        "top5_accuracy": top5_correct / total if total > 0 else 0.0,
    }


def compute_per_sample_perplexity(model, tokenizer, texts, device,
                                   max_length: int = MAX_LENGTH) -> np.ndarray:
    """
    Return per-sample perplexity for statistical comparison.
    Each element is the perplexity of one story.
    """
    model.eval()
    per_sample = []
    with torch.no_grad():
        for text in texts:
            enc  = tokenizer(text, return_tensors="pt", truncation=True,
                             max_length=max_length)
            ids  = enc["input_ids"].to(device)
            mask = enc["attention_mask"].to(device)
            lbl  = ids.clone()
            lbl[mask == 0] = -100
            out  = model(input_ids=ids, attention_mask=mask, labels=lbl)
            per_sample.append(float(torch.exp(out.loss).item()))
    return np.array(per_sample)


def compute_ts_metrics(model_path, eval_texts, device, tokenizer) -> Dict:
    """Load a model and compute all TinyStories evaluation metrics."""
    hf_model = GPT2LMHeadModel.from_pretrained(model_path).to(device)

    ppl    = compute_perplexity(hf_model, tokenizer, eval_texts, device)
    acc    = compute_next_token_accuracy(hf_model, tokenizer, eval_texts, device)
    pp_arr = compute_per_sample_perplexity(hf_model, tokenizer, eval_texts, device)

    hf_model.cpu()
    del hf_model
    torch.cuda.empty_cache()

    return {
        "perplexity":     ppl,
        "top1_accuracy":  acc["top1_accuracy"],
        "top5_accuracy":  acc["top5_accuracy"],
        "ppl_per_sample": pp_arr,
    }


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def run_ts_ablation_study(imp_texts: List[str], eval_texts: List[str]):
    """
    Main entry point.

    Parameters
    ----------
    imp_texts  : texts used to compute gradient importance scores
    eval_texts : held-out texts used for evaluation metrics

    Usage in Colab
    --------------
        from ablation_tinystories import run_ts_ablation_study, load_tinystories
        imp_texts, eval_texts = load_tinystories()
        results = run_ts_ablation_study(imp_texts, eval_texts)
    """
    for d in [ABLATION_SAVE_DIR, CACHE_DIR, RESULTS_DIR]:
        os.makedirs(d, exist_ok=True)

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # ── Base model metrics ────────────────────────────────────
    print("\nEvaluating base model...")
    base_metrics = compute_ts_metrics(
        os.path.join(EXISTING_MODELS_DIR, "base_model"),
        eval_texts, DEVICE, tokenizer)

    base_ppl      = base_metrics["perplexity"]
    base_top1     = base_metrics["top1_accuracy"]
    base_top5     = base_metrics["top5_accuracy"]
    base_ppl_arr  = base_metrics["ppl_per_sample"]

    print(f"  Base PPL:           {base_ppl:.2f}")
    print(f"  Base Top-1 Acc:     {base_top1*100:.2f}%")
    print(f"  Base Top-5 Acc:     {base_top5*100:.2f}%")

    # ── Importance scores on TinyStories ─────────────────────
    print("\nComputing importance scores on TinyStories...")
    base_hf = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)

    ts_dataset = TinyStoriesDataset(imp_texts, tokenizer)
    ts_loader  = DataLoader(ts_dataset, batch_size=8, shuffle=False)

    layer_scores_heads = compute_layer_gradient_importance(
        base_hf, ts_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_heads_ts.pt"))

    layer_scores_neurons = compute_layer_gradient_importance_neurons(
        base_hf, ts_loader, DEVICE,
        cache_path=os.path.join(CACHE_DIR, "grad_importance_neurons_ts.pt"))

    base_hf.cpu()
    del base_hf
    torch.cuda.empty_cache()

    run_ts_head_diagnostic(layer_scores_heads)

    # ── Variants ──────────────────────────────────────────────
    # Structure mirrors IOI/GT scripts exactly.
    # use_protected_heads=True has no effect here since TS_PROTECTED_HEADS
    # is empty — included for structural parity and future extension.
    variants = [
        # ── Attribution – attention heads ──
        dict(name="attribution_attention_heads_ts",
             label="Attribution TS – full (attn heads)",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=True,
             use_protected_heads=True),
        dict(name="ablation_ts_no_budgets",
             label="Attribution TS – no layer budgets (attn)",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=False,
             use_protected_heads=True),
        dict(name="ablation_ts_no_protection",
             label="Attribution TS – no head protection",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=True,
             use_protected_heads=False),
        dict(name="ablation_ts_no_budgets_no_protection",
             label="Attribution TS – no budgets no protection",
             load_existing=False,
             component="attention_heads",
             use_layer_budgets=False,
             use_protected_heads=False),

        # ── Attribution – neurons ──
        dict(name="attribution_neurons_ts",
             label="Attribution TS – full (neurons)",
             load_existing=False,
             component="neurons",
             use_layer_budgets=True),
        dict(name="ablation_ts_neurons_no_budgets",
             label="Attribution TS – no layer budgets (neurons)",
             load_existing=False,
             component="neurons",
             use_layer_budgets=False),

        # ── Baselines (reuse saved models from IOI/GT runs) ──
        dict(name="magnitude_attention_heads",
             label="Magnitude – attn heads",
             load_existing=True,
             component="attention_heads"),
        dict(name="magnitude_neurons",
             label="Magnitude – neurons",
             load_existing=True,
             component="neurons"),
        dict(name="random_attention_heads",
             label="Random – attn heads",
             load_existing=True,
             component="attention_heads"),
        dict(name="random_neurons",
             label="Random – neurons",
             load_existing=True,
             component="neurons"),
    ]

    all_results = []

    for v in variants:
        print(f"\n{'='*60}")
        print(f"Variant: {v['label']}")
        print(f"{'='*60}")

        for sparsity_pct in SPARSITY_LEVELS:
            model_path = get_model_path(v["name"], sparsity_pct)
            print(f"  Path:   {model_path}")
            print(f"  Exists: {os.path.exists(model_path)}")

            # Build new models if not already saved
            if not v["load_existing"] and not os.path.exists(model_path):
                print(f"\n  Pruning sparsity {sparsity_pct}%...")
                base_fresh = GPT2LMHeadModel.from_pretrained("gpt2")

                if v["component"] == "attention_heads":
                    pruned = prune_attention_heads_ablation(
                        base_fresh, layer_scores_heads, sparsity_pct,
                        use_layer_budgets=v["use_layer_budgets"],
                        use_protected_heads=v["use_protected_heads"],
                    )
                else:
                    pruned = prune_neurons_ablation(
                        base_fresh, layer_scores_neurons, sparsity_pct,
                        use_layer_budgets=v["use_layer_budgets"],
                    )

                os.makedirs(model_path, exist_ok=True)
                pruned.save_pretrained(model_path)
                tokenizer.save_pretrained(model_path)
                del base_fresh, pruned
                torch.cuda.empty_cache()
                print(f"    Saved to {model_path}")

            if not os.path.exists(model_path):
                print(f"  ⚠️  Not found: {model_path} — skipping")
                continue

            print(f"\n  Evaluating sparsity {sparsity_pct}%...")

            actual  = get_actual_sparsity(model_path)
            metrics = compute_ts_metrics(model_path, eval_texts, DEVICE, tokenizer)

            ppl      = metrics["perplexity"]
            top1     = metrics["top1_accuracy"]
            top5     = metrics["top5_accuracy"]
            ppl_arr  = metrics["ppl_per_sample"]

            # Per-sample Cohen's d vs base (higher d = more degradation)
            d = cohens_d(ppl_arr, base_ppl_arr, paired=True)

            # Statistical test: is perplexity significantly higher than base?
            from scipy.stats import ttest_rel
            _, p_val = ttest_rel(ppl_arr, base_ppl_arr)

            # Bootstrap CI on perplexity delta
            delta_arr = ppl_arr - base_ppl_arr
            ci        = bootstrap_confidence_interval(delta_arr)

            row = {
                "variant":             v["label"],
                "nominal_sparsity":    sparsity_pct,
                "actual_sparsity_pct": round(actual * 100, 1),
                "perplexity":          round(ppl, 2),
                "ppl_delta":           round(ppl - base_ppl, 2),
                "ppl_delta_pct":       round((ppl - base_ppl) / base_ppl * 100, 1),
                "top1_accuracy_pct":   round(top1 * 100, 2),
                "top1_delta_pct":      round((top1 - base_top1) / base_top1 * 100, 1),
                "top5_accuracy_pct":   round(top5 * 100, 2),
                "cohens_d_ppl":        round(d, 3),
                "p_value_ttest":       round(float(p_val), 4),
                "ci_delta_lower":      round(float(ci[0]), 3),
                "ci_delta_upper":      round(float(ci[1]), 3),
            }
            all_results.append(row)

            print(f"    Actual sparsity:   {actual*100:.1f}%")
            print(f"    Perplexity:        {ppl:.2f}  (Δ {ppl - base_ppl:+.2f} / {row['ppl_delta_pct']:+.1f}%)")
            print(f"    Top-1 Accuracy:    {top1*100:.2f}%  (Δ {row['top1_delta_pct']:+.1f}%)")
            print(f"    Top-5 Accuracy:    {top5*100:.2f}%")
            print(f"    Cohen's d (PPL):   {d:.3f}")
            print(f"    p-value (t-test):  {p_val:.4f}")

    # ── Save results ──────────────────────────────────────────
    if not all_results:
        print("No results to save.")
        return all_results

    csv_path  = os.path.join(RESULTS_DIR, "ablation_results_ts.csv")
    json_path = os.path.join(RESULTS_DIR, "base_metrics_ts.json")

    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)
    print(f"\n✓ Results saved to {csv_path}")

    with open(json_path, "w") as f:
        json.dump({
            "perplexity":    base_ppl,
            "top1_accuracy": base_top1,
            "top5_accuracy": base_top5,
        }, f, indent=2)

    print_table(all_results, base_ppl, base_top1)
    return all_results


# ─────────────────────────────────────────────────────────────
# TABLE
# ─────────────────────────────────────────────────────────────

def print_table(results, base_ppl, base_top1):
    print(f"\n{'='*110}")
    print("TINYSTORIES ABLATION STUDY RESULTS")
    print(f"Base: PPL={base_ppl:.2f} | Top-1 Acc={base_top1*100:.2f}%")
    print(f"{'='*110}")

    for sparsity_pct in SPARSITY_LEVELS:
        rows = [r for r in results if r["nominal_sparsity"] == sparsity_pct]
        print(f"\n── {sparsity_pct}% Nominal Sparsity ──")
        print(f"{'Variant':<45} {'Act%':>5} {'PPL':>8} {'PPLΔ%':>7} "
              f"{'Top-1%':>8} {'Top-1Δ%':>9} {'d(PPL)':>8} {'p':>7}")
        print("-" * 105)
        for r in rows:
            print(f"{r['variant']:<45} "
                  f"{r['actual_sparsity_pct']:>5.1f} "
                  f"{r['perplexity']:>8.2f} "
                  f"{r['ppl_delta_pct']:>6.1f}% "
                  f"{r['top1_accuracy_pct']:>7.2f}% "
                  f"{r['top1_delta_pct']:>8.1f}% "
                  f"{r['cohens_d_ppl']:>8.3f} "
                  f"{r['p_value_ttest']:>7.4f}")


# ─────────────────────────────────────────────────────────────
# USAGE
# ─────────────────────────────────────────────────────────────


In [ ]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — TinyStories Ablation Pre-flight Check
# Paste into a cell in your existing notebook and run.
# All functions assumed to already be defined above.
# Expected runtime: ~3-5 minutes on T4.
# ─────────────────────────────────────────────────────────────

PASS = "✅"
FAIL = "❌"
_results = []

def _check(name, fn):
    try:
        out = fn()
        print(f"{PASS} {name}: {out}")
        _results.append((name, True))
    except Exception as e:
        print(f"{FAIL} {name}: {e}")
        _results.append((name, False))


# ── 1. TinyStories loads ──────────────────────────────────────
print("=" * 55)
print("CHECK 1 — TinyStories data loading")
print("=" * 55)

_ts_texts = None

def _load_ts():
    global _ts_texts
    from datasets import load_dataset
    ds = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
    _ts_texts = [ex["text"] for ex in list(ds.take(20))]
    assert len(_ts_texts) == 20
    return f"{len(_ts_texts)} stories, first 50 chars: '{_ts_texts[0][:50]}...'"

_check("TinyStories load", _load_ts)


# ── 2. Tokenizer + dataset wrapper ───────────────────────────
print("\n" + "=" * 55)
print("CHECK 2 — Tokenizer + TinyStoriesDataset")
print("=" * 55)

def _dataset_check():
    from torch.utils.data import DataLoader
    ds     = TinyStoriesDataset(_ts_texts[:10], tokenizer, max_length=64)
    loader = DataLoader(ds, batch_size=2)
    batch  = next(iter(loader))
    assert batch["input_ids"].shape == (2, 64)
    assert batch["labels"].shape    == (2, 64)
    return f"Batch shape OK: {tuple(batch['input_ids'].shape)}"

_check("TinyStoriesDataset", _dataset_check)


# ── 3. Base model perplexity ──────────────────────────────────
print("\n" + "=" * 55)
print("CHECK 3 — Base model perplexity")
print("=" * 55)

_base_model_path = os.path.join(EXISTING_MODELS_DIR, "base_model")

def _base_ppl():
    m   = GPT2LMHeadModel.from_pretrained(_base_model_path).to(DEVICE)
    ppl = compute_perplexity(m, tokenizer, _ts_texts[:10], DEVICE)
    m.cpu(); del m; torch.cuda.empty_cache()
    assert 1 < ppl < 1000, f"PPL {ppl:.2f} out of expected range"
    return f"PPL = {ppl:.2f}"

_check("Base model perplexity", _base_ppl)


# ── 4. Gradient importance scores ────────────────────────────
print("\n" + "=" * 55)
print("CHECK 4 — Gradient importance (5 stories)")
print("=" * 55)

_layer_scores_heads   = None
_layer_scores_neurons = None

def _importance():
    global _layer_scores_heads, _layer_scores_neurons
    from torch.utils.data import DataLoader
    m      = GPT2LMHeadModel.from_pretrained(_base_model_path).to(DEVICE)
    ds     = TinyStoriesDataset(_ts_texts[:5], tokenizer, max_length=64)
    loader = DataLoader(ds, batch_size=2, shuffle=False)

    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)

    _layer_scores_heads   = compute_layer_gradient_importance(m, loader, DEVICE)
    _layer_scores_neurons = compute_layer_gradient_importance_neurons(m, loader, DEVICE)
    m.cpu(); del m; torch.cuda.empty_cache()

    for l in range(12):
        assert _layer_scores_heads[l].sum() > 0,            f"L{l} head scores all zero"
        assert _layer_scores_neurons[l].sum() > 0,          f"L{l} neuron scores all zero"
        assert torch.isfinite(_layer_scores_heads[l]).all(), f"L{l} head scores has NaN/Inf"
        assert torch.isfinite(_layer_scores_neurons[l]).all(),f"L{l} neuron scores has NaN/Inf"

    top_head = _layer_scores_heads[11].argmax().item()
    return f"Scores OK — L11 most important head: H{top_head}"

_check("Gradient importance scores", _importance)


# ── 5. Pruning ────────────────────────────────────────────────
print("\n" + "=" * 55)
print("CHECK 5 — Build attribution model (attn heads, 30%)")
print("=" * 55)

_pruned_model = None

def _pruning():
    global _pruned_model
    base = GPT2LMHeadModel.from_pretrained(_base_model_path)
    _pruned_model = prune_attention_heads_ablation(
        base, _layer_scores_heads,
        sparsity_pct=30,
        use_layer_budgets=True,
        use_protected_heads=True,
    )
    del base
    total = zeros = 0
    for p in _pruned_model.parameters():
        total += p.numel()
        zeros += (p.data == 0).sum().item()
    actual = zeros / total
    assert actual > 0,   "Pruned model has zero sparsity"
    assert actual < 0.5, f"Sparsity {actual:.1%} too high for 30% nominal"
    return f"Actual sparsity: {actual*100:.1f}%"

_check("Pruning (attn heads, 30%)", _pruning)


# ── 6. Evaluate pruned model ──────────────────────────────────
print("\n" + "=" * 55)
print("CHECK 6 — Evaluate pruned model")
print("=" * 55)

def _eval_pruned():
    assert _pruned_model is not None, "Pruned model not built (check 5 failed)"
    _pruned_model.to(DEVICE)
    ppl = compute_perplexity(_pruned_model, tokenizer, _ts_texts[:10], DEVICE)
    acc = compute_next_token_accuracy(_pruned_model, tokenizer, _ts_texts[:10], DEVICE)
    _pruned_model.cpu()
    torch.cuda.empty_cache()
    assert ppl > 0 and torch.isfinite(torch.tensor(ppl))
    return (f"PPL = {ppl:.2f} | "
            f"Top-1 = {acc['top1_accuracy']*100:.1f}% | "
            f"Top-5 = {acc['top5_accuracy']*100:.1f}%")

_check("Evaluate pruned model", _eval_pruned)


# ── 7. Baseline models accessible ────────────────────────────
print("\n" + "=" * 55)
print("CHECK 7 — Baseline models accessible")
print("=" * 55)

for _variant in ["magnitude_attention_heads", "magnitude_neurons",
                 "random_attention_heads",    "random_neurons"]:
    for _sparsity in SPARSITY_LEVELS:
        _path = os.path.join(EXISTING_MODELS_DIR, f"{_variant}_sparsity{_sparsity}")
        def _check_path(p=_path, v=_variant, s=_sparsity):
            assert os.path.exists(p), f"Not found: {p}"
            files = os.listdir(p)
            assert any(f.endswith(".bin") or f.endswith(".safetensors")
                       for f in files), f"No weights in {p}"
            return f"Found ({len(files)} files)"
        _check(f"{_variant} @ {_sparsity}%", _check_path)


# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print("DIAGNOSTIC SUMMARY")
print("=" * 55)
_passed = sum(1 for _, ok in _results if ok)
_total  = len(_results)
print(f"{_passed}/{_total} checks passed\n")

for _name, _ok in _results:
    print(f"  {'✅' if _ok else '❌'} {_name}")

if _passed == _total:
    print("\n🟢 All checks passed — safe to run full ablation.")
else:
    print("\n🔴 Fix the following before running:")
    for _name, _ok in _results:
        if not _ok:
            print(f"    - {_name}")

CHECK 1 — TinyStories data loading
✅ TinyStories load: 20 stories, first 50 chars: 'Spot. Spot saw the shiny car and said, "Wow, Kitty...'

CHECK 2 — Tokenizer + TinyStoriesDataset
✅ TinyStoriesDataset: Batch shape OK: (2, 64)

CHECK 3 — Base model perplexity


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Base model perplexity: PPL = 10.90

CHECK 4 — Gradient importance (5 stories)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Computing gradient × activation importance on TinyStories...


...

  Computing gradient × activation importance on TinyStories...


  Gradient batches: 100%|██████████| 3/3 [00:01<00:00,  2.66it/s]


✅ Gradient importance scores: Scores OK — L11 most important head: H8

CHECK 5 — Build attribution model (attn heads, 30%)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Pruning (attn heads, 30%): Actual sparsity: 2.1%

CHECK 6 — Evaluate pruned model
✅ Evaluate pruned model: PPL = 110.98 | Top-1 = 29.7% | Top-5 = 53.3%

CHECK 7 — Baseline models accessible
✅ magnitude_attention_heads @ 30%: Found (7 files)
✅ magnitude_attention_heads @ 50%: Found (7 files)
✅ magnitude_attention_heads @ 70%: Found (7 files)
✅ magnitude_neurons @ 30%: Found (7 files)
✅ magnitude_neurons @ 50%: Found (7 files)
✅ magnitude_neurons @ 70%: Found (7 files)
✅ random_attention_heads @ 30%: Found (7 files)
✅ random_attention_heads @ 50%: Found (7 files)
✅ random_attention_heads @ 70%: Found (7 files)
✅ random_neurons @ 30%: Found (7 files)
✅ random_neurons @ 50%: Found (7 files)
✅ random_neurons @ 70%: Found (7 files)

DIAGNOSTIC SUMMARY
18/18 checks passed

  ✅ TinyStories load
  ✅ TinyStoriesDataset
  ✅ Base model perplexity
  ✅ Gradient importance scores
  ✅ Pruning (attn heads, 30%)
  ✅ Evaluate pruned model
  ✅ magnitude_attention_heads @ 30%
  ✅ magnitude_attention_head

In [ ]:
# Load data
imp_texts, eval_texts = load_tinystories(n_importance=500, n_eval=200)

# Run ablation
results = run_ts_ablation_study(imp_texts, eval_texts)


Evaluating base model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  Base PPL:           11.06
  Base Top-1 Acc:     47.20%
  Base Top-5 Acc:     74.48%

Computing importance scores on TinyStories...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

    Actual sparsity:   22.8%
    Perplexity:        291.15  (Δ +280.09 / +2533.4%)
    Top-1 Accuracy:    13.54%  (Δ -71.3%)
    Top-5 Accuracy:    30.17%
    Cohen's d (PPL):   3.488
    p-value (t-test):  0.0000
  Path:   /content/drive/MyDrive/IRP/GPT-2/pruned_models/random_neurons_sparsity70
  Exists: True

  Evaluating sparsity 70%...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

    Actual sparsity:   31.9%
    Perplexity:        1102.67  (Δ +1091.61 / +9873.6%)
    Top-1 Accuracy:    5.72%  (Δ -87.9%)
    Top-5 Accuracy:    17.73%
    Cohen's d (PPL):   4.120
    p-value (t-test):  0.0000

✓ Results saved to /content/drive/MyDrive/IRP/ablation_results_ts/ablation_results_ts.csv

TINYSTORIES ABLATION STUDY RESULTS
Base: PPL=11.06 | Top-1 Acc=47.20%

── 30% Nominal Sparsity ──
Variant                                        Act%      PPL   PPLΔ%   Top-1%   Top-1Δ%   d(PPL)       p
---------------------------------------------------------------------------------------------------------
Attribution TS – full (attn heads)              2.1    14.26   29.0%   42.77%     -9.4%    2.408  0.0000
Attribution TS – no layer budgets (attn)        5.7   259.32 2245.5%   22.23%    -52.9%    1.622  0.0000
Attribution TS – no head protection             2.1    14.26   29.0%   42.77%     -9.4%    2.408  0.0000
Attribution TS – no budgets no protection       5.7   259.32 2245.5% 

#Sparsity Computation

In [ ]:
def compute_all_sparsities(results_csv_path=None):
    """
    Read actual sparsity from every saved model across all ablation dirs.
    Prints a clean table grouped by variant and sparsity level.
    """
    import os
    from transformers import GPT2LMHeadModel

    model_dirs = {
        "IOI ablation":       "/content/drive/MyDrive/IRP/ablation_variants_v5",
        "GT ablation":        "/content/drive/MyDrive/IRP/ablation_variants_gt",
        "TS ablation":        "/content/drive/MyDrive/IRP/ablation_variants_ts",
        "Baselines":          "/content/drive/MyDrive/IRP/GPT-2/pruned_models",
    }

    def get_sparsity(model_path):
        m = GPT2LMHeadModel.from_pretrained(model_path)
        total = zeros = 0
        for p in m.parameters():
            total += p.numel()
            zeros += (p.data == 0).sum().item()
        del m
        return zeros / total

    print(f"\n{'='*65}")
    print(f"{'Model Path':<45} {'Actual Sparsity':>15}")
    print(f"{'='*65}")

    all_results = {}

    for group_name, base_dir in model_dirs.items():
        if not os.path.exists(base_dir):
            print(f"\n⚠️  Not found: {base_dir}")
            continue

        print(f"\n── {group_name} ({base_dir}) ──")

        subdirs = sorted(os.listdir(base_dir))
        for subdir in subdirs:
            model_path = os.path.join(base_dir, subdir)
            if not os.path.isdir(model_path):
                continue
            if not os.path.exists(os.path.join(model_path, "config.json")):
                continue

            try:
                sparsity = get_sparsity(model_path)
                print(f"  {subdir:<45} {sparsity*100:>6.2f}%")
                all_results[subdir] = sparsity
            except Exception as e:
                print(f"  {subdir:<45} ERROR: {e}")

    print(f"\n{'='*65}")
    print("SPARSITY COMPARISON — Same nominal level, different methods")
    print(f"{'='*65}")

    for sparsity_pct in [30, 50, 70]:
        tag = f"sparsity{sparsity_pct}"
        rows = {k: v for k, v in all_results.items() if tag in k}
        if not rows:
            continue
        print(f"\n── Nominal {sparsity_pct}% ──")
        print(f"  {'Variant':<50} {'Actual%':>8}")
        print(f"  {'-'*60}")
        for name, sp in sorted(rows.items(), key=lambda x: x[1]):
            print(f"  {name:<50} {sp*100:>7.2f}%")

    return all_results

sparsities = compute_all_sparsities()


Model Path                                    Actual Sparsity

── IOI ablation (/content/drive/MyDrive/IRP/ablation_variants_v5) ──


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  ablation_neurons_no_budgets_sparsity30         13.65%


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  ablation_neurons_no_budgets_sparsity50         22.77%


...

  random_neurons_sparsity30                      13.65%


Loading weights:   0%|          | 0/148 [00:01<?, ?it/s]

  random_neurons_sparsity50                      22.77%


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  random_neurons_sparsity70                      31.87%

SPARSITY COMPARISON — Same nominal level, different methods

── Nominal 30% ──
  Variant                                             Actual%
  ------------------------------------------------------------
  ablation_no_protection_sparsity30                     2.05%
  attribution_attention_heads_sparsity30                2.05%
  ablation_gt_no_protection_sparsity30                  2.05%
  attribution_attention_heads_gt_sparsity30             2.05%
  ablation_ts_no_protection_sparsity30                  2.05%
  attribution_attention_heads_ts_sparsity30             2.05%
  ablation_no_budgets_no_protection_sparsity30          5.69%
  ablation_no_budgets_sparsity30                        5.69%
  ablation_gt_no_budgets_no_protection_sparsity30       5.69%
  ablation_gt_no_budgets_sparsity30                     5.69%
  ablation_ts_no_budgets_no_protection_sparsity30       5.69%
  ablation_ts_no_budgets_sparsity30                     5

In [ ]:
import os
import torch
import numpy as np
from copy import deepcopy
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformer_lens import HookedTransformer
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


N_LAYERS  = 12
N_HEADS   = 12
HEAD_DIM  = 64
SPARSITY  = 30
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

# ── FIX 1: Use your actual CKA-derived budgets, not the wrong ones ─────────
LAYER_BUDGETS_ATTN = {
    0: 0.4,
    1: 0.4,
    2: 0.4,
    3: 0.4,
    4: 0.4,
    5: 0.4,
    6: 0.4,
    7: 0.4,
    8: 0.6,
    9: 0.5,
    10: 0.4,
    11: 0.3,
}

IOI_PROTECTED_HEADS = {
    (8, 6), (8, 10),
    (9, 6), (9, 9),
    (10, 0), (10, 7),
    (11, 10),
}


GT_PROTECTED_HEADS = {
    (5, 0), (7, 10), (7, 11), (8, 6), (9, 1),
}

MAX_HEADS_PER_LAYER = {
        0: 2, 1: 2, 2: 2, 3: 2,
        4: 3, 5: 3, 6: 3, 7: 3,
        8: 2, 9: 1, 10: 1, 11: 1,
    }

# ── TinyStories wrapper ────────────────────────────────────────────────────
class TinyStoriesWrapper(Dataset):
    def __init__(self, texts, tokenizer, max_length=64):
        self.encodings = []
        for text in texts:
            enc = tokenizer(text, truncation=True, max_length=max_length,
                           padding="max_length", return_tensors="pt")
            self.encodings.append(enc)

    def __len__(self): return len(self.encodings)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings[idx]["input_ids"].squeeze(0),
            "attention_mask": self.encodings[idx]["attention_mask"].squeeze(0),
        }


def get_tinystories_texts(n=500):
    ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
    return [x["text"] for x, _ in zip(ds, range(n))]


# ── Importance scoring ─────────────────────────────────────────────────────
def compute_gradient_importance_heads(hf_model, dataloader, device):
    print("  Computing gradient × activation importance...")
    importance = {i: torch.zeros(N_HEADS) for i in range(N_LAYERS)}
    hf_model.train()
    n_batches = 0

    for batch in tqdm(dataloader, desc="  Grad batches"):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        hf_model.zero_grad()
        out  = hf_model(input_ids=ids, attention_mask=mask, labels=ids)
        out.loss.backward()

        with torch.no_grad():
            for layer_idx in range(N_LAYERS):
                W = hf_model.transformer.h[layer_idx].attn.c_proj.weight
                G = hf_model.transformer.h[layer_idx].attn.c_proj.weight.grad
                if G is None:
                    continue
                score = (W * G).abs()
                for head_idx in range(N_HEADS):
                    s = head_idx * HEAD_DIM
                    e = s + HEAD_DIM
                    importance[layer_idx][head_idx] += score[:, s:e].sum().item()

        n_batches += 1
        if n_batches >= 63:
            break

    hf_model.eval()
    for layer_idx in range(N_LAYERS):
        importance[layer_idx] /= n_batches
    return importance


def compute_magnitude_importance_heads(hf_model):
    print("  Computing magnitude importance...")
    importance = {}
    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            W = hf_model.transformer.h[layer_idx].attn.c_proj.weight
            scores = torch.zeros(N_HEADS)
            for head_idx in range(N_HEADS):
                s = head_idx * HEAD_DIM
                e = s + HEAD_DIM
                scores[head_idx] = W[:, s:e].norm(p=2)
            importance[layer_idx] = scores
    return importance


# ── FIX 2: Pruning now matches your existing ablation exactly ──────────────
# Zeros both c_attn and c_proj slices, uses MAX_HEADS_PER_LAYER cap,
# and handles protected heads as a set of (layer, head) tuples.
def prune_heads(base_model, importance, sparsity_pct,
                use_budgets=True, protected_heads=None):
    if protected_heads is None:
        protected_heads = set()

    nominal = sparsity_pct / 100.0
    pruned  = deepcopy(base_model)

    HIDDEN_SIZE = N_HEADS * HEAD_DIM  # 768

    with torch.no_grad():
        for layer_idx in range(N_LAYERS):
            budget = LAYER_BUDGETS_ATTN[layer_idx] if use_budgets else 1.0
            eff    = nominal * budget

            if use_budgets:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    MAX_HEADS_PER_LAYER[layer_idx],
                    int(N_HEADS * 0.40),
                )
            else:
                n_prune = min(
                    max(0, int(N_HEADS * eff)),
                    int(N_HEADS * 0.70),
                )

            if n_prune == 0:
                continue

            scores = importance[layer_idx].clone()
            # Protected heads get pushed to the back (won't be pruned)
            for (l, h) in protected_heads:
                if l == layer_idx:
                    scores[h] = float('inf')

            ranked = sorted(range(N_HEADS), key=lambda h: scores[h].item())

            attn = pruned.transformer.h[layer_idx].attn
            pruned_count = 0
            for h in ranked:
                if pruned_count >= n_prune:
                    break
                if scores[h] == float('inf'):
                    break
                # Zero c_attn (Q, K, V projections) and c_proj slices
                s = h * HEAD_DIM
                e = s + HEAD_DIM
                attn.c_attn.weight[:, s:e] = 0
                attn.c_attn.weight[:, HIDDEN_SIZE + s:HIDDEN_SIZE + e] = 0
                attn.c_attn.weight[:, 2 * HIDDEN_SIZE + s:2 * HIDDEN_SIZE + e] = 0
                attn.c_proj.weight[s:e, :] = 0
                pruned_count += 1

    return pruned


# ── Metrics ────────────────────────────────────────────────────────────────
def compute_actual_sparsity(model):
    total = zeros = 0
    for p in model.parameters():
        total += p.numel()
        zeros += (p.data == 0).sum().item()
    return zeros / total


def compute_ioi_accuracy_from_dataset(hf_model, ioi_dataset, device):
    hf_model.eval()
    correct = total = 0

    with torch.no_grad():
        for i in range(len(ioi_dataset.toks)):
            ids     = ioi_dataset.toks[i].unsqueeze(0).to(device)
            end_pos = ioi_dataset.word_idx["end"][i].item()
            io_tok  = ioi_dataset.io_tokenIDs[i].item()
            s_tok   = ioi_dataset.s_tokenIDs[i].item()

            out    = hf_model(ids)
            logits = out.logits[0, end_pos, :]

            if logits[io_tok] > logits[s_tok]:
                correct += 1
            total += 1

    return correct / total if total > 0 else 0.0


# ── FIX 3: Pass tokenizer explicitly instead of accessing gt_dataset.tokenizer
def compute_gt_accuracy_from_dataset(hf_model, gt_dataset, tokenizer, device):
    hf_model.eval()
    correct = total = 0

    with torch.no_grad():
        for i in range(len(gt_dataset.toks)):
            ids     = gt_dataset.toks[i].unsqueeze(0).to(device)
            end_pos = gt_dataset.word_idx["end"][i].item()
            yy      = gt_dataset.years_YY[i].item()

            out    = hf_model(ids)
            logits = out.logits[0, end_pos, :]

            valid_token_ids = []
            for candidate_yy in range(yy + 1, 100):
                yy_str  = f"{candidate_yy:02d}"
                tok_ids = tokenizer(yy_str, add_special_tokens=False)["input_ids"]
                if len(tok_ids) == 1:
                    valid_token_ids.append(tok_ids[0])

            if not valid_token_ids:
                continue

            predicted = logits.argmax().item()
            if predicted in valid_token_ids:
                correct += 1
            total += 1

    return correct / total if total > 0 else 0.0


# ── FIX 4: compute_perplexity now accepts eval_texts directly ──────────────
def compute_perplexity(hf_model, tokenizer, eval_texts, device,
                       max_length=128):
    hf_model.eval()
    total_loss = total_toks = 0

    with torch.no_grad():
        for text in eval_texts:
            ids = tokenizer(text, return_tensors="pt",
                           truncation=True,
                           max_length=max_length)["input_ids"].to(device)
            if ids.shape[1] < 2:
                continue
            out = hf_model(ids, labels=ids)
            total_loss += out.loss.item() * (ids.shape[1] - 1)
            total_toks += ids.shape[1] - 1

    return np.exp(total_loss / total_toks) if total_toks > 0 else float('nan')


def evaluate_all(hf_model, label, ioi_dataset, gt_dataset,
                 tokenizer, eval_texts, device):
    sparsity = compute_actual_sparsity(hf_model)
    ioi_acc  = compute_ioi_accuracy_from_dataset(hf_model, ioi_dataset, device)
    gt_acc   = compute_gt_accuracy_from_dataset(hf_model, gt_dataset,
                                                 tokenizer, device)
    ppl      = compute_perplexity(hf_model, tokenizer, eval_texts, device)

    print(f"\n  [{label}]")
    print(f"    Actual sparsity: {sparsity*100:.2f}%")
    print(f"    IOI accuracy:    {ioi_acc*100:.1f}%")
    print(f"    GT accuracy:     {gt_acc*100:.1f}%")
    print(f"    Perplexity:      {ppl:.1f}")

    return {"label": label, "sparsity": sparsity,
            "ioi_acc": ioi_acc, "gt_acc": gt_acc, "ppl": ppl}


# ── Main ───────────────────────────────────────────────────────────────────
def main():
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    # ── Eval texts (held-out, not used for importance scoring) ────────────
    print("Loading eval texts...")
    ds = load_dataset("roneneldan/TinyStories", split="validation", streaming=True)
    eval_texts = [ex["text"] for ex in list(ds.take(200))]
    print(f"  {len(eval_texts)} eval stories loaded")

    # ── TL model for dataset creation ─────────────────────────────────────
    print("\nLoading TransformerLens model...")
    tl_model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    tl_model.tokenizer.pad_token = tl_model.tokenizer.eos_token

    # ── Datasets ──────────────────────────────────────────────────────────
    print("\nCreating IOI dataset...")
    ioi_dataset, _ = create_ioi_datasets(tl_model, num_samples=500, seed=42)

    print("\nCreating GT dataset...")
    gt_dataset, _  = create_greater_than_datasets(tl_model, num_samples=500, seed=42)

    # ── TinyStories dataloader for gradient scoring (separate from eval) ──
    print("\nLoading TinyStories for gradient reference...")
    ts_texts  = get_tinystories_texts(500)
    ts_ds     = TinyStoriesWrapper(ts_texts, tokenizer)
    ts_loader = DataLoader(ts_ds, batch_size=8, shuffle=False)

    ioi_texts_for_grad = list(ioi_dataset.sentences)
    ioi_ds     = TinyStoriesWrapper(ioi_texts_for_grad, tokenizer)
    ioi_loader = DataLoader(ioi_ds, batch_size=8, shuffle=False)

    gt_texts_for_grad = list(gt_dataset.sentences)
    gt_ds     = TinyStoriesWrapper(gt_texts_for_grad, tokenizer)
    gt_loader = DataLoader(gt_ds, batch_size=8, shuffle=False)

    # ── Importance scores ─────────────────────────────────────────────────
    print("\n" + "="*60)
    print("Computing importance scores")
    print("="*60)

    hf_scratch = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    mag_imp    = compute_magnitude_importance_heads(hf_scratch)
    del hf_scratch; torch.cuda.empty_cache()

    # hf_scratch   = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    # grad_ioi_imp = compute_gradient_importance_heads(hf_scratch, ioi_loader, DEVICE)
    # del hf_scratch; torch.cuda.empty_cache()

    # hf_scratch  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    # grad_gt_imp = compute_gradient_importance_heads(hf_scratch, gt_loader, DEVICE)
    # del hf_scratch; torch.cuda.empty_cache()

    # hf_scratch  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    # grad_ts_imp = compute_gradient_importance_heads(hf_scratch, ts_loader, DEVICE)
    # del hf_scratch; torch.cuda.empty_cache()

    # Load cached IOI importance from original ablation (guarantees identical scoring)
    print("  Loading cached IOI importance from original ablation...")
    cached_ioi = torch.load(
        "/content/drive/MyDrive/IRP/ablation_cache_v5/grad_importance_heads.pt",
        map_location="cpu"
    )
    grad_ioi_imp = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        imp = cached_ioi[key]
        grad_ioi_imp[layer_idx] = torch.tensor([
            imp[h*HEAD_DIM:(h+1)*HEAD_DIM, :].mean().item()
            for h in range(N_HEADS)
        ])
    del cached_ioi

    print("  Loading cached GT importance from original ablation...")
    cached_gt = torch.load(
        "/content/drive/MyDrive/IRP/ablation_cache_gt/grad_importance_heads_gt.pt",
        map_location="cpu"
    )
    grad_gt_imp = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        imp = cached_gt[key]
        grad_gt_imp[layer_idx] = torch.tensor([
            imp[h*HEAD_DIM:(h+1)*HEAD_DIM, :].mean().item()
            for h in range(N_HEADS)
        ])
    del cached_gt

    print("  Loading cached TS importance from original ablation...")
    cached_ts = torch.load(
        "/content/drive/MyDrive/IRP/ablation_cache_ts/grad_importance_heads_ts.pt",
        map_location="cpu"
    )
    grad_ts_imp = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        imp = cached_ts[key]
        grad_ts_imp[layer_idx] = torch.tensor([
            imp[h*HEAD_DIM:(h+1)*HEAD_DIM, :].mean().item()
            for h in range(N_HEADS)
        ])
    del cached_ts

    # ── Baseline ──────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("BASELINE (unpruned)")
    print("="*60)
    base_hf  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
    base_res = evaluate_all(base_hf, "Base (unpruned)",
                            ioi_dataset, gt_dataset,
                            tokenizer, eval_texts, DEVICE)
    base_hf.cpu(); del base_hf; torch.cuda.empty_cache()
    all_results = [base_res]

    # ── Variants ──────────────────────────────────────────────────────────
    variants = [
        ("Attribution IOI + budgets + protection",
         grad_ioi_imp, True,  IOI_PROTECTED_HEADS),
        ("Attribution GT  + budgets + protection",
         grad_gt_imp,  True,  GT_PROTECTED_HEADS),
        ("Attribution TS  + budgets",
         grad_ts_imp,  True,  set()),

        # Controlled: same budgets + protection, magnitude scoring
        ("Magnitude + CKA budgets + IOI protection",
         mag_imp,      True,  IOI_PROTECTED_HEADS),
        ("Magnitude + CKA budgets + GT  protection",
         mag_imp,      True,  GT_PROTECTED_HEADS),
        ("Magnitude + CKA budgets (no protection)",
         mag_imp,      True,  set()),

        # Existing baseline for reference
        ("Magnitude baseline (no budgets)",
         mag_imp,      False, set()),
    ]

    for label, imp, use_budgets, protected in variants:
        print(f"\n{'='*60}")
        print(f"Variant: {label}")
        print(f"{'='*60}")

        fresh  = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
        pruned = prune_heads(fresh, imp, SPARSITY,
                             use_budgets=use_budgets,
                             protected_heads=protected)
        fresh.cpu(); del fresh; torch.cuda.empty_cache()

        pruned.to(DEVICE)
        res = evaluate_all(pruned, label, ioi_dataset, gt_dataset,
                           tokenizer, eval_texts, DEVICE)
        all_results.append(res)
        pruned.cpu(); del pruned; torch.cuda.empty_cache()

    # ── Summary ───────────────────────────────────────────────────────────
    print("\n\n" + "="*80)
    print(f"SUMMARY — Nominal {SPARSITY}% Sparsity")
    print(f"Base: IOI {base_res['ioi_acc']*100:.1f}% | "
          f"GT {base_res['gt_acc']*100:.1f}% | PPL {base_res['ppl']:.1f}")
    print("="*80)
    print(f"{'Variant':<45} {'Act%':>6} {'IOI':>7} {'GT':>7} {'PPL':>8}")
    print("-"*80)
    for r in all_results:
        print(f"  {r['label']:<43} {r['sparsity']*100:>5.2f}% "
              f"{r['ioi_acc']*100:>6.1f}% {r['gt_acc']*100:>6.1f}% "
              f"{r['ppl']:>8.1f}")
    print("="*80)

    # ── Key comparison ────────────────────────────────────────────────────
    print("\nKEY QUESTION: Does gradient scoring add value beyond budget design?")
    print("-"*60)

    pairs = [
        ("Attribution IOI + budgets + protection",
         "Magnitude + CKA budgets + IOI protection",
         "IOI circuit", "ioi_acc"),
        ("Attribution GT  + budgets + protection",
         "Magnitude + CKA budgets + GT  protection",
         "GT circuit", "gt_acc"),
        ("Attribution TS  + budgets",
         "Magnitude + CKA budgets (no protection)",
         "TinyStories", "ppl"),
    ]

    for attr_label, mag_label, circuit, metric in pairs:
        attr = next((r for r in all_results if r["label"] == attr_label), None)
        mag  = next((r for r in all_results if r["label"] == mag_label),  None)
        if not attr or not mag:
            continue

        if metric == "ppl":
            diff   = mag[metric] - attr[metric]
            better = "Attribution better ✓" if diff > 0 else "Magnitude better"
            print(f"\n  {circuit} (PPL, lower is better):")
            print(f"    Attribution + budgets: {attr[metric]:.1f}")
            print(f"    Magnitude   + budgets: {mag[metric]:.1f}")
        else:
            diff   = (attr[metric] - mag[metric]) * 100
            better = "Attribution better ✓" if diff > 0 else "Magnitude better"
            print(f"\n  {circuit} (Accuracy, higher is better):")
            print(f"    Attribution + budgets: {attr[metric]*100:.1f}%")
            print(f"    Magnitude   + budgets: {mag[metric]*100:.1f}%")

        print(f"    Difference: {diff:+.1f} → {better}")

    print("\nDone.")


if __name__ == "__main__":
    main()

Loading eval texts...
  200 eval stories loaded

Loading TransformerLens model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded pretrained model gpt2 into HookedTransformer

Creating IOI dataset...
45

[VALIDATION] Checking end positions for first 3 examples:
  Example 1:
    Full sentence: Since Ruby and Peter were at the park, Peter offered a book to Ruby
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Ruby' (position 15)
    Expected IO:     ' Ruby'
    ✅ CORRECT: Model will predict IO name
  Example 2:
    Full sentence: After Kimberly and Frank arrived at the park, Frank passed a watch to Kimberly
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Kimberly' (position 15)
    Expected IO:     ' Kimberly'
    ✅ CORRECT: Model will predict IO name
  Example 3:
    Full sentence: When Noah and Frank walked into the store, Noah handed a book to Frank
    End position: 14
    Token AT end:    ' to' (position 14)
    Token AFTER end: ' Frank' (position 15)
    Expected IO:     ' Frank'
    ✅ CORRECT: Model will predict IO name

✓ Create

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  [Magnitude + CKA budgets (no protection)]
    Actual sparsity: 2.05%
    IOI accuracy:    60.0%
    GT accuracy:     74.2%
    Perplexity:      15.2

Variant: Magnitude baseline (no budgets)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  [Magnitude baseline (no budgets)]
    Actual sparsity: 5.69%
    IOI accuracy:    37.6%
    GT accuracy:     1.8%
    Perplexity:      40.9


SUMMARY — Nominal 30% Sparsity
Base: IOI 95.0% | GT 94.6% | PPL 11.0
Variant                                         Act%     IOI      GT      PPL
--------------------------------------------------------------------------------
  Base (unpruned)                              0.00%   95.0%   94.6%     11.0
  Attribution IOI + budgets + protection       2.05%   82.2%   86.8%     61.6
  Attribution GT  + budgets + protection       2.05%   87.8%   96.6%     55.0
  Attribution TS  + budgets                    2.05%   93.4%   86.6%     14.2
  Magnitude + CKA budgets + IOI protection     2.05%   60.0%   74.2%     15.2
  Magnitude + CKA budgets + GT  protection     2.05%   60.0%   74.2%     15.2
  Magnitude + CKA budgets (no protection)      2.05%   60.0%   74.2%     15.2
  Magnitude baseline (no budgets)              5.69%   37.6%    1.8%     40.9

KE

In [ ]:
def load_and_process_head_importance(cache_path):
    raw = torch.load(cache_path, map_location="cpu")
    layer_scores = {}
    for layer_idx in range(N_LAYERS):
        key = f"transformer.h.{layer_idx}.attn.c_proj.weight"
        if key in raw:
            imp = raw[key]
            scores = torch.tensor([
                imp[h * HEAD_DIM:(h + 1) * HEAD_DIM, :].mean().item()
                for h in range(N_HEADS)
            ])
        else:
            scores = torch.zeros(N_HEADS)
        layer_scores[layer_idx] = scores
    return layer_scores

layer_scores_heads_ts = load_and_process_head_importance(
    "/content/drive/MyDrive/IRP/ablation_cache_ts/grad_importance_heads_ts.pt")

layer_scores_heads_gt = load_and_process_head_importance(
    "/content/drive/MyDrive/IRP/ablation_cache_gt/grad_importance_heads_gt.pt")

# Verify
print(type(list(layer_scores_heads_ts.keys())[0]))  # should be <class 'int'>
print(layer_scores_heads_ts[0].shape)               # should be torch.Size([12])

<class 'int'>
torch.Size([12])
